# Step 5 — Internal validation and negative control

**Lymphoma AI Research** (`dlbcl-prognosis`) · locked analysis plan v1.0, sections 8–9 · code version 05-1.0

Research project only. Nothing in this notebook is a diagnostic or treatment tool.

Each stage starts only after the previous one has passed:

1. Plan lock, input files and the step-4b manifest are verified (sections 2–6).
2. **Reproducibility check (training set only, section 7).** The step-4b fit is run again and must give the same model as recorded in `manifest_04b.json`. This proves that the permuted models below are fitted with exactly the procedure that produced the frozen model.
3. **Leakage check (training set only, plan §9.2, section 8).** Nested CV on 10 label-permuted training sets. If the mean C is above 0.55, the notebook stops and the test set stays closed.
4. **Frozen model check (plan §8, section 9).** The SHA-256 of `model_04.json` is compared with `manifest_04b.json` and with the value recorded in `progress_step4b.md`; its genes, coefficients, training statistics and cutoff are checked against the training data.
5. **Null models (training set only, plan §9.1, section 10).** The whole procedure is re-run on 100 label-permuted training sets. These models are fixed before the test set is opened.
6. **Test set (sections 11–14).** Opened for the first time in section 11: H1 and the other metrics of §8.2, the negative control and the two sensitivity analyses.
7. Red flags, the pre-specified conclusion, output files, `manifest_05.json` and a summary to paste back (sections 15–16).

TR: Bu not defteri kilitli planın 8. ve 9. bölümlerini uygular. Sıra: (1) 4b'deki model eğitim verisinden yeniden kurulur; bu, kodun 4b ile aynı sonucu verdiğinin kanıtıdır; (2) eğitim setinde sızıntı kontrolü yapılır; (3) geçerse dondurulmuş modelin SHA-256 özeti manifestoyla karşılaştırılır; (4) negatif kontrolün 100 permütasyon modeli yalnızca eğitim verisiyle kurulur; (5) test seti ancak bunların hepsinden sonra, 11. bölümde açılır.

**Süre:** Colab'da yaklaşık 1 saat (50–75 dakika); en uzun kısımlar 8. ve 10. bölümlerdeki permütasyonlar. Biten her iş Drive'a kaydedilir. Bağlantı koparsa **Çalışma zamanı → Tümünü çalıştır**'a yeniden bas; biten kısımlar yeniden hesaplanmaz. Tableti şarja tak ve ekran zaman aşımını en uzun süreye ayarla, çünkü ekran kapanınca tarayıcı bağlantıyı kesebilir.

Bittiğinde en alttaki özeti sohbete yapıştır.

## 1. Setup
TR: Gerekirse `scikit-survival` ve `lifelines` paketlerini 4b'deki sürümleriyle kurar, Drive'ı bağlar, klasörleri tanımlar ve paket sürümlerini yazdırır. Sürümler 2. bölümde 4b'ninkilerle karşılaştırılır; asıl kontrol 7. bölümdeki yeniden üretim testidir.

In [ ]:
# Install the two survival packages only if this runtime does not have them yet (same versions as step 4b).
import importlib.metadata, importlib.util, subprocess, sys
_PINNED = {'scikit-survival': ('sksurv', '0.28.0'), 'lifelines': ('lifelines', '0.30.3')}
_missing = [f'{pkg}=={ver}' for pkg, (mod, ver) in _PINNED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    _dist = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
    _loaded = {m: sys.modules[m].__version__ for m in _dist if m in sys.modules}
    print('Installing', ', '.join(_missing), '(about a minute)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=True)
    _changed = [m for m, v in _loaded.items() if importlib.metadata.version(_dist[m]) != v]
    if _changed:
        raise RuntimeError(f'pip changed packages that were already loaded ({_changed}). '
                           'Choose Runtime -> Restart session, then Run all again.')

import datetime, hashlib, json, os, pickle, platform, re, tempfile, time, warnings
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, matplotlib
import matplotlib.pyplot as plt
import sksurv, lifelines
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import median_survival_times

PROJECT_DIR = os.environ.get('LYMPHOMA_PROJECT_DIR', '/content/drive/MyDrive/dlbcl-prognosis')
if PROJECT_DIR.startswith('/content/drive') and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
assert os.path.isdir(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'

CLEAN, PROC = f'{PROJECT_DIR}/data/clean', f'{PROJECT_DIR}/data/processed'
RES4 = f'{PROJECT_DIR}/results/04'
RES, FIG = f'{PROJECT_DIR}/results/05', f'{PROJECT_DIR}/figures/05'
CKPT = f'{RES}/checkpoints_05'
for _d in (RES, FIG, CKPT):
    os.makedirs(_d, exist_ok=True)

N_JOBS = max(1, os.cpu_count() or 1)
JOBLIB_TMP = os.path.join(tempfile.gettempdir(), 'joblib_05')      # local disk of the runtime, not Drive
os.makedirs(JOBLIB_TMP, exist_ok=True)
T_START, RUNTIME = time.time(), {}

VERSIONS = {'python': sys.version.split()[0], 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
            'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit-learn': sklearn.__version__, 'scikit-survival': sksurv.__version__,
            'lifelines': lifelines.__version__, 'joblib': joblib.__version__, 'matplotlib': matplotlib.__version__}
for k, v in VERSIONS.items():
    print(f'{k:16s} {v}')
print('parallel workers:', N_JOBS)

## 2. Plan lock, input files and the step-4b manifest
The lock file must be the one recorded after 04a, the two plan files must match the hashes stored in it, and `plan_amendments.md` must have no entries. All 12 input files are hashed and compared with the lock. The step-4b manifest must belong to the same lock and report no red flags. The model file itself is checked in section 9, after the leakage check. The external cohorts are only hashed (they belong to step 6).

TR: Kilit dosyasını, iki plan dosyasını ve 12 girdi dosyasının hepsini SHA-256 ile kontrol eder; bir şey değiştiyse durur. 4b manifestosunun aynı kilide ait olduğunu ve kırmızı bayrak içermediğini doğrular. Model dosyasının kendisi sızıntı kontrolünden sonra, 9. bölümde kontrol edilir. Dış kohortların dosyaları açılmaz, yalnızca özetleri alınır.

In [ ]:
EXPECTED_LOCK_SHA256 = '8d87cf887e2db4770ae3273755d4901cd0d5a2a010fdc729159e007aa129bc7d'    # progress_step4a.md
EXPECTED_MODEL_SHA256 = '661e2b80be774252a7fd613116ecc06673a8617ed8a2230bdae56170e3908403'   # progress_step4b.md (section 9)
CODE_VERSION = '05-1.0'

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

LOCK_PATH = f'{RES4}/plan_lock_04.json'
assert os.path.exists(LOCK_PATH), 'results/04/plan_lock_04.json not found.'
LOCK_SHA = sha256_file(LOCK_PATH)
assert LOCK_SHA == EXPECTED_LOCK_SHA256, (
    f'The lock file is not the one recorded after 04a ({EXPECTED_LOCK_SHA256[:12]} expected, {LOCK_SHA[:12]} found). Stop.')
with open(LOCK_PATH) as f:
    LOCK = json.load(f)
for kind in ('json', 'md'):
    path = f"{PROJECT_DIR}/{LOCK[f'plan_{kind}_file']}"
    assert sha256_file(path) == LOCK[f'plan_{kind}_sha256'], f'{path} differs from the locked version. Stop.'
with open(f"{PROJECT_DIR}/{LOCK['plan_json_file']}") as f:
    PLAN = json.load(f)
assert PLAN['version'] == '1.0'

AMEND_PATH = f"{PROJECT_DIR}/{PLAN['deviations']['file']}"
amend_text = open(AMEND_PATH).read() if os.path.exists(AMEND_PATH) else ''
AMENDMENTS = re.findall(r'^## Amendment \d+.*$', amend_text, flags=re.M)
if AMENDMENTS:
    raise RuntimeError(f'plan_amendments.md has {len(AMENDMENTS)} entries. This notebook implements plan v1.0 '
                       'without amendments; check before running it.')

ACCESS = {'clinical_all': 'opened: training rows (section 3), test rows (section 11)',
          'split': 'opened (section 3)',
          'qc_samples': 'opened: training rows (section 3), test rows (section 11)',
          'expr_train': 'opened (section 3)',
          'gene_table': 'opened (section 3)',
          'expr_test': 'hashed now; opened only in section 11, after the gates'}
INPUT_SHA = {}
print('Input files (SHA-256 compared with the lock):')
for key, rel in LOCK['input_files'].items():
    path = f'{PROJECT_DIR}/{rel}'
    assert os.path.exists(path), f'missing input file: {rel}'
    h = sha256_file(path)
    INPUT_SHA[key] = {'file': rel, 'sha256': h, 'status': 'match' if h == LOCK['input_sha256'][key] else 'CHANGED'}
    print(f"  {INPUT_SHA[key]['status']:8s} {rel:46s} ({ACCESS.get(key, 'hashed only, never opened')})")
CHANGED = [k for k, v in INPUT_SHA.items() if v['status'] != 'match']
assert not CHANGED, f'Input files changed since the plan was locked: {CHANGED}. Stop.'
assert set(ACCESS) <= set(INPUT_SHA), 'the lock does not list the expected input files'

M4B_PATH = f'{RES4}/manifest_04b.json'
assert os.path.exists(M4B_PATH), 'results/04/manifest_04b.json not found. Run 04b first.'
with open(M4B_PATH) as f:
    M4B = json.load(f)
assert M4B['plan']['lock_file_sha256'] == LOCK_SHA, 'manifest_04b.json belongs to a different plan lock. Stop.'
assert (M4B['plan']['plan_json_sha256'] == LOCK['plan_json_sha256']
        and M4B['plan']['plan_md_sha256'] == LOCK['plan_md_sha256'] and M4B['plan']['amendments'] == 0)
_bad = [k for k, v in M4B['inputs_verified_against_lock'].items() if v['sha256'] != LOCK['input_sha256'].get(k)]
assert not _bad, f'manifest_04b.json lists input hashes that differ from the lock: {_bad}. Stop.'
assert not M4B['red_flags'], f"Step 4b reported red flags: {M4B['red_flags']}. Review them before step 5."
print(f"Plan v{PLAN['version']} lock verified ({LOCK_SHA[:12]}); no amendments; all {len(INPUT_SHA)} input files unchanged.")
print(f"manifest_04b.json ({M4B['created_utc']}) belongs to this lock; no red flags in step 4b.")
_vdiff = {k: (VERSIONS[k], v) for k, v in M4B['environment'].items()
          if k in VERSIONS and k not in ('platform', 'cpu_count') and str(VERSIONS[k]) != str(v)}
print('Package versions: same as in step 4b.' if not _vdiff else
      f'Package versions differ from step 4b (now, step 4b): {_vdiff}. Section 7 checks that the fit is still reproduced.')

## 3. Training data (primary analysis population)
Only the training rows of the clinical and QC tables are kept; the other rows are dropped right after loading, and the test expression file is not opened. Samples flagged `sex_mismatch_confident` are excluded (plan §2). The counts must equal those recorded in `manifest_04b.json`. Patients are sorted by `sample_id`, because the CV fold assignment depends on the row order.

TR: Yalnızca eğitim satırlarını yükler, kilitli QC kuralını uygular (7 örnek çıkar) ve sayıları 4b manifestosuyla karşılaştırır: 373 hasta, 171 ölüm. Test satırları yüklenir yüklenmez bellekten silinir; test ifade dosyası bu aşamada açılmaz.

In [ ]:
def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'yes'])

def gene_id_text(v):
    s = str(v).strip()
    return s.split('.')[0] if re.fullmatch(r'\d+\.0+', s) else s

TIME, EVENT = PLAN['endpoints']['primary']['time'], PLAN['endpoints']['primary']['event']
FLAG_P, FLAG_S = PLAN['qc']['primary_exclusion_flag'], PLAN['qc']['sensitivity_exclusion_flag']
TRN, TST = PLAN['cohorts']['training'], PLAN['cohorts']['internal_test']

split = pd.read_csv(f'{CLEAN}/split_GSE181063.csv')
assert split['sample_id'].is_unique, 'duplicate sample IDs in the split file'
train_ids = set(split.loc[split['set'] == TRN['split'], 'sample_id'])
test_ids = set(split.loc[split['set'] == TST['split'], 'sample_id'])      # IDs only; no test data are loaded here
assert len(train_ids) == TRN['patients'] and len(test_ids) == TST['patients'] and not (train_ids & test_ids), \
    'the split file does not match the plan'

clin_all = pd.read_csv(f'{CLEAN}/clinical_clean_all.csv')
tr = clin_all[clin_all['sample_id'].isin(train_ids)].copy()
del clin_all                                        # test and external rows leave memory here
qc_all = pd.read_csv(f'{PROC}/qc_samples.csv')
qc_flags = qc_all.loc[qc_all['sample_id'].isin(train_ids), ['sample_id', FLAG_P, FLAG_S]].copy()
del qc_all
assert qc_flags['sample_id'].is_unique and set(qc_flags['sample_id']) == train_ids, 'QC rows do not match the training set'
qc_flags[FLAG_P], qc_flags[FLAG_S] = as_bool(qc_flags[FLAG_P]), as_bool(qc_flags[FLAG_S])

assert tr['sample_id'].is_unique and len(tr) == TRN['patients'] and int(tr[EVENT].sum()) == TRN['deaths'], \
    'training counts differ from the plan'
tr = tr.merge(qc_flags, on='sample_id', how='left', validate='one_to_one')
POP4B = M4B['population']
EXCLUDED = sorted(tr.loc[tr[FLAG_P], 'sample_id'])
assert EXCLUDED == sorted(POP4B['excluded_by_primary_qc']), 'QC exclusions differ from step 4b'
pop = tr.loc[~tr[FLAG_P]].sort_values('sample_id').reset_index(drop=True)

T = pop[TIME].to_numpy(dtype=float)
E = pop[EVENT].to_numpy(dtype=int)
assert np.isfinite(T).all() and (T > 0).all() and set(np.unique(E)) <= {0, 1}
CORR_OUT = pop[FLAG_S].to_numpy(dtype=bool)
SAMPLE_IDS = pop['sample_id'].to_numpy()
counts = {'patients': (len(pop), POP4B['patients']), 'deaths': (int(E.sum()), POP4B['deaths']),
          'ipi_missing': (int(pop['ipi'].isna().sum()), POP4B['ipi_missing']),
          'corr_outliers': (int(CORR_OUT.sum()), POP4B['corr_outliers_retained'])}
_bad = {k: v for k, v in counts.items() if v[0] != v[1]}
assert not _bad, f'Primary training population differs from step 4b: {_bad}'

expr = pd.read_parquet(f'{PROC}/expr_GSE181063_train.parquet')
if 'gene' in expr.columns:
    expr = expr.set_index('gene')
assert set(expr.columns) == train_ids, 'expression columns are not exactly the training samples'
assert expr.shape[0] == PLAN['features']['n_genes_input'], 'unexpected number of genes'
GENE_INDEX_DTYPE = str(expr.index.dtype)
GENE_STR = np.array([gene_id_text(g) for g in expr.index])
assert len(set(GENE_STR)) == len(GENE_STR), 'duplicate gene IDs'
assert all(re.fullmatch(r'\d+', g) for g in GENE_STR), 'the gene index does not look like Entrez IDs'
X = expr[list(SAMPLE_IDS)].T.to_numpy(dtype=np.float64)        # patients x genes, rows in pop order
del expr
assert np.isfinite(X).all(), 'missing or infinite expression values'

gt = pd.read_csv(f'{PROC}/gene_table_common.csv', dtype=str)
gt['gene'] = gt['gene'].map(gene_id_text)
gt = gt.drop_duplicates('gene').set_index('gene')
SYM_COLS = [c for c in ('symbol_GPL570', 'symbol') if c in gt.columns]

def symbol_of(g):
    for c in SYM_COLS:
        s = gt[c].get(g)
        if isinstance(s, str) and s.strip():
            return s.strip()
    return g

SYMBOLS = np.array([symbol_of(g) for g in GENE_STR])
_vals, _cnt = np.unique(T, return_counts=True)
print(f'Primary training population: {len(pop)} patients, {E.sum()} deaths ({len(EXCLUDED)} excluded by {FLAG_P}); '
      f'all counts equal manifest_04b.json.')
print(f'Correlation outliers kept: {CORR_OUT.sum()}; tied OS times: {(_cnt > 1).sum()} values, {_cnt[_cnt > 1].sum()} patients.')
print(f"Expression: {X.shape[1]:,} genes (Entrez IDs, index dtype {GENE_INDEX_DTYPE}). Test IDs known: {len(test_ids)}; "
      'no test data loaded.')

## 4. Model-fitting functions (step 4b, unchanged)
The functions below are the step-4b code, character for character. `fit_procedure` carries out plan §4–5 on any dataset: variance filter and z-score estimated on that dataset, scikit-survival's penalty grid, 10-fold CV stratified by event with the grouped partial-likelihood deviance of glmnet, λ_min and λ_1se. The permuted-label models of sections 8 and 10 must be fitted with exactly the procedure that produced the frozen model; sections 7 and 9 prove that by reproducing it.

TR: 4b'deki model kurma kodu, tek karakteri değişmeden. Permütasyon modelleri dondurulmuş modeli üreten yöntemin aynısıyla kurulmalı; bunu 7. ve 9. bölümler modeli yeniden üreterek gösterir.

In [ ]:
# Core numerical functions for step 4b. Pure functions: every input is passed explicitly,
# so they also run unchanged inside the parallel worker processes.
import time
import warnings

import numpy as np
from joblib import Parallel, delayed
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

NUMERICAL_ERRORS = (ArithmeticError, AssertionError)


# ---------------------------------------------------------------- likelihood and metrics
def breslow_loglik(eta, time_, event):
    """Breslow log partial likelihood; one value per column of eta (n x L)."""
    eta = np.asarray(eta, dtype=float)
    if eta.ndim == 1:
        eta = eta[:, None]
    time_ = np.asarray(time_, dtype=float)
    order = np.argsort(-time_, kind='mergesort')                    # descending time
    t_s, e_s, eta_s = time_[order], np.asarray(event).astype(bool)[order], eta[order]
    log_cum = np.logaddexp.accumulate(eta_s, axis=0)                # log sum exp over {time >= t}
    _, first, counts = np.unique(-t_s, return_index=True, return_counts=True)
    last_of_block = np.repeat(first + counts - 1, counts)           # tied times share one risk set
    return (eta_s[e_s] - log_cum[last_of_block][e_s]).sum(axis=0)


def breslow_saturated_loglik(time_, event):
    t_events = np.asarray(time_, dtype=float)[np.asarray(event).astype(bool)]
    _, d = np.unique(t_events, return_counts=True)
    return float(-np.sum(d * np.log(d)))


def breslow_deviance(eta, time_, event):
    """2 x (saturated - model) log partial likelihood, Breslow ties, unit weights.
    The quantity glmnet's coxnet.deviance() returns (computed there by the coxdev library)."""
    return 2.0 * (breslow_saturated_loglik(time_, event) - breslow_loglik(eta, time_, event))


def harrell_c(time_, event, score):
    """Harrell's C (higher score = higher risk). All-tied scores give 0.5."""
    return float(concordance_index_censored(np.asarray(event).astype(bool),
                                            np.asarray(time_, dtype=float),
                                            np.asarray(score, dtype=float))[0])


# ---------------------------------------------------------------- pre-processing (plan section 4)
def preprocess_fit(X, top_n, ddof=1):
    """Variance filter (top_n genes; all genes if top_n is None) and z-score statistics,
    estimated on the rows of the current fit only."""
    var = X.var(axis=0, ddof=ddof)
    if top_n is None or top_n >= X.shape[1]:
        idx = np.arange(X.shape[1])
    else:
        idx = np.sort(np.argsort(-var, kind='stable')[:top_n])
    mean = X[:, idx].mean(axis=0)
    sd = X[:, idx].std(axis=0, ddof=ddof)
    return {'idx': idx, 'mean': mean, 'sd': np.where(sd > 0, sd, 1.0), 'n_constant': int((sd <= 0).sum())}


def transform(X, prep):
    return (X[:, prep['idx']] - prep['mean']) / prep['sd']


# ---------------------------------------------------------------- penalty grid
def alpha_grid_from_max(alpha_max, n_alphas, alpha_min_ratio):
    """Same arithmetic as scikit-survival's automatic grid (repeated multiplication)."""
    factor = alpha_min_ratio ** (1.0 / (n_alphas - 1.0))
    grid = np.empty(n_alphas)
    grid[0] = alpha_max
    for i in range(1, n_alphas):
        grid[i] = grid[i - 1] * factor
    return grid


def breslow_alpha_max(Z, time_, event, l1_ratio):
    """Smallest penalty with all coefficients zero: max |score at beta = 0| / (n * l1_ratio).
    Used only if scikit-survival's automatic path fails numerically."""
    time_ = np.asarray(time_, dtype=float)
    order = np.argsort(-time_, kind='mergesort')
    t_s, e_s, Z_s = time_[order], np.asarray(event).astype(bool)[order], Z[order]
    _, first, counts = np.unique(-t_s, return_index=True, return_counts=True)
    last = np.repeat(first + counts - 1, counts)
    xbar = np.cumsum(Z_s, axis=0)[last] / (last + 1.0)[:, None]
    score = (Z_s[e_s] - xbar[e_s]).sum(axis=0)
    return float(np.max(np.abs(score)) / (Z.shape[0] * l1_ratio))


# ---------------------------------------------------------------- path fitting
def _coxnet(Z, y, l1_ratio, alphas, cfg):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        est = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alphas=alphas, n_alphas=cfg['n_alphas'],
                                     alpha_min_ratio=cfg['alpha_min_ratio'],
                                     max_iter=cfg['max_iter'], tol=cfg['tol']).fit(Z, y)
    return est, sum(issubclass(w.category, ConvergenceWarning) for w in caught)


def _fit_segment(Z, y, l1_ratio, seg, cfg):
    """One call along seg. On a numerical failure, keep the longest prefix of seg that fits
    (a prefix of a warm-started path is computed identically, so the search is valid)."""
    try:
        est, n_conv = _coxnet(Z, y, l1_ratio, seg, cfg)
        return est, n_conv, None
    except NUMERICAL_ERRORS:
        lo, hi, best = 0, len(seg), (None, 0)
        while hi - lo > 1:
            mid = (lo + hi) // 2
            try:
                best = _coxnet(Z, y, l1_ratio, seg[:mid], cfg)
                lo = mid
            except NUMERICAL_ERRORS:
                hi = mid
        return best[0], best[1], lo                    # lo = first failing position within seg


def new_path_info():
    return {'restarts_after_spurious_stop': 0, 'early_stop_at': None, 'failed_from': None,
            'convergence_warnings': 0}


def fit_path(Z, time_, event, l1_ratio, grid, cfg):
    """Coefficient path on a fixed decreasing grid. Returns coef (p x L; NaN after a failure),
    the number of valid penalties and a dict of events along the path.

    scikit-survival stops a path when the deviance ratio changes by < 0.1 % over 5 penalties.
    With tied times its null deviance ignores ties, so the deviance ratio starts slightly below 0
    and the rule can fire spuriously; such stops (deviance ratio < 0 at the stop) are restarted at
    the next penalty. Genuine stops (deviance ratio > 0): later penalties take the last fitted
    coefficients, as glmnet does for a fold path shorter than the lambda sequence."""
    L, p = len(grid), Z.shape[1]
    coef = np.zeros((p, L))
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    info = new_path_info()
    pos = 0
    while pos < L:
        seg = grid[pos:]
        est, n_conv, fail_in_seg = _fit_segment(Z, y, l1_ratio, seg, cfg)
        info['convergence_warnings'] += n_conv
        if est is None:                                # the first penalty of the segment fails
            info['failed_from'] = pos
            break
        k = est.coef_.shape[1]
        coef[:, pos:pos + k] = est.coef_
        end = pos + k
        target = len(seg) if fail_in_seg is None else fail_in_seg
        if k < target:                                 # stopped by scikit-survival's rule
            if est.deviance_ratio_[k - 1] < 0:
                info['restarts_after_spurious_stop'] += 1
                pos = end
                continue
            info['early_stop_at'] = end
            fill_to = L if fail_in_seg is None else pos + fail_in_seg
            coef[:, end:fill_to] = coef[:, [end - 1]]
            if fail_in_seg is not None:
                info['failed_from'] = pos + fail_in_seg
            break
        if fail_in_seg is not None:                    # reached the first failing penalty
            info['failed_from'] = end
            break
        pos = end
    n_valid = L if info['failed_from'] is None else info['failed_from']
    coef[:, n_valid:] = np.nan
    return coef, n_valid, info


def auto_path(Z, time_, event, l1_ratio, cfg):
    """Full-data fit: scikit-survival's automatic grid (cfg['n_alphas'] values) and the path."""
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    try:
        est, n_conv = _coxnet(Z, y, l1_ratio, None, cfg)
        alpha_max = float(est.alphas_[0])
    except NUMERICAL_ERRORS:
        est, n_conv, alpha_max = None, 0, breslow_alpha_max(Z, time_, event, l1_ratio)
    grid = alpha_grid_from_max(alpha_max, cfg['n_alphas'], cfg['alpha_min_ratio'])
    if est is not None:
        k = len(est.alphas_)
        assert np.allclose(est.alphas_, grid[:k], rtol=1e-10, atol=0), 'grid reconstruction mismatch'
        if k == len(grid):
            info = new_path_info()
            info['convergence_warnings'] = n_conv
            return grid, np.array(est.coef_, dtype=float), len(grid), info
    coef, n_valid, info = fit_path(Z, time_, event, l1_ratio, grid, cfg)
    return grid, coef, n_valid, info


# ---------------------------------------------------------------- cross-validation (plan section 5)
def run_parallel(tasks, n_jobs, temp_folder):
    if n_jobs == 1:
        return [f(*a, **kw) for f, a, kw in tasks]
    return Parallel(n_jobs=n_jobs, temp_folder=temp_folder, max_nbytes='1M')(tasks)


def cv_fold_task(X_cur, time_, event, train_idx, grid, l1_ratio, top_n, cfg):
    """One CV fold: pre-processing and path on the fold's training rows; Breslow deviance of every
    path coefficient on all rows of the current fit and on the fold's training rows."""
    X_tr = np.asarray(X_cur[train_idx])
    prep = preprocess_fit(X_tr, top_n, cfg['ddof'])
    coef, n_valid, info = fit_path(transform(X_tr, prep), time_[train_idx], event[train_idx],
                                   l1_ratio, grid, cfg)
    dev_diff = np.full(len(grid), np.nan)
    if n_valid > 0:
        eta = transform(np.asarray(X_cur), prep) @ coef[:, :n_valid]
        dev_diff[:n_valid] = (breslow_deviance(eta, time_, event)
                              - breslow_deviance(eta[train_idx], time_[train_idx], event[train_idx]))
    return {'dev_diff': dev_diff, 'n_valid': n_valid, 'info': info}


def cv_select(X_cur, time_, event, grid, n_valid_full, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder):
    """Grouped cross-validated partial-likelihood deviance, as in glmnet (cv.coxnet, cvstats,
    getOptcv): per fold (D_full - D_train) / fold size; fold-size-weighted mean and SE.
    Penalties at or after a numerical failure in any fit are dropped for all folds (plan section 5)."""
    folds = list(StratifiedKFold(n_splits=cfg['cv_folds'], shuffle=True,
                                 random_state=cv_seed).split(np.zeros(len(event)), event))
    res = run_parallel([delayed(cv_fold_task)(X_cur, time_, event, tr, grid, l1_ratio, top_n, cfg)
                        for tr, _ in folds], n_jobs, temp_folder)
    n_valid = min([n_valid_full] + [r['n_valid'] for r in res])
    if n_valid == 0:
        raise RuntimeError('every penalty failed numerically in at least one fit')
    w = np.array([len(te) for _, te in folds], dtype=float)
    cvraw = np.vstack([r['dev_diff'][:n_valid] for r in res]) / w[:, None]
    cvm = (w[:, None] * cvraw).sum(axis=0) / w.sum()
    cvsd = np.sqrt((w[:, None] * (cvraw - cvm) ** 2).sum(axis=0) / w.sum() / (len(folds) - 1))
    idx_min = int(np.flatnonzero(cvm <= cvm.min())[0])                 # largest lambda at the minimum
    idx_1se = int(np.flatnonzero(cvm <= cvm[idx_min] + cvsd[idx_min])[0])
    return {'cvm': cvm, 'cvsd': cvsd, 'idx_min': idx_min, 'idx_1se': idx_1se, 'n_valid': n_valid,
            'fold_info': [r['info'] for r in res], 'fold_sizes': w.astype(int).tolist()}


def fit_procedure(X_cur, time_, event, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder):
    """The whole procedure of plan sections 4-5 on one dataset (the 'current fit')."""
    t0 = time.time()
    X_cur = np.asarray(X_cur, dtype=float)
    time_, event = np.asarray(time_, dtype=float), np.asarray(event).astype(int)
    prep = preprocess_fit(X_cur, top_n, cfg['ddof'])
    grid, coef, n_valid_full, info_full = auto_path(transform(X_cur, prep), time_, event, l1_ratio, cfg)
    cv = cv_select(X_cur, time_, event, grid, n_valid_full, l1_ratio, top_n, cv_seed, cfg, n_jobs, temp_folder)
    n_genes_path = (np.abs(np.nan_to_num(coef)) > 0).sum(axis=0)
    return {'prep': prep, 'grid': grid, 'coef': coef, 'n_valid_full': n_valid_full, 'info_full': info_full,
            'n_genes_path': n_genes_path, 'n_patients': len(event), 'n_deaths': int(event.sum()),
            'l1_ratio': l1_ratio, 'top_n': top_n, 'cv_seed': cv_seed, 'seconds': time.time() - t0, **cv}


def model_at(proc, idx):
    """Selected genes (column positions in X), coefficients and z-score statistics at grid index idx."""
    b = proc['coef'][:, idx]
    sel = np.flatnonzero(b != 0)
    return {'gene_pos': proc['prep']['idx'][sel], 'coef': b[sel], 'mean': proc['prep']['mean'][sel],
            'sd': proc['prep']['sd'][sel], 'lambda': float(proc['grid'][idx]), 'idx': int(idx)}


def lp_of(model, X_rows):
    X_rows = np.asarray(X_rows, dtype=float)
    if len(model['coef']) == 0:
        return np.zeros(X_rows.shape[0])
    return ((X_rows[:, model['gene_pos']] - model['mean']) / model['sd']) @ model['coef']


def path_events(proc):
    """Counts of path events in the full-data fit and in all CV folds of one procedure."""
    infos = [proc['info_full']] + list(proc['fold_info'])
    return {'spurious_stop_restarts': int(sum(i['restarts_after_spurious_stop'] for i in infos)),
            'genuine_early_stops': int(sum(i['early_stop_at'] is not None for i in infos)),
            'numerical_failures': int(sum(i['failed_from'] is not None for i in infos)),
            'convergence_warnings': int(sum(i['convergence_warnings'] for i in infos)),
            'penalties_dropped': int(len(proc['grid']) - proc['n_valid'])}


# Self-check: Breslow log partial likelihood against its definition (direct loop over risk sets, with ties).
_rng = np.random.default_rng(0)
_t = np.round(_rng.exponential(30, 60)); _e = (_rng.random(60) < 0.6).astype(int); _eta = _rng.normal(0, 3, 60)
_ll = sum(_eta[i] - np.log(np.exp(_eta[_t >= _t[i]]).sum()) for i in np.flatnonzero(_e))
assert np.isclose(breslow_loglik(_eta, _t, _e)[0], _ll, rtol=1e-12)
print('likelihood self-check passed')

## 5. Validation functions (plan §8.2)
One function, `evaluate()`, computes every metric of §8.2 for one risk score in one analysis population, so the primary analysis and the sensitivity analyses run the same code:
- Cox HR per 1 SD of the LP (SD of the population's own LP) with 95% CI — the primary metric;
- Harrell's C with a percentile bootstrap CI; Uno's C truncated at τ (rule of §8.2);
- Kaplan–Meier by risk group, log-rank test, HR high vs low;
- calibration slope (Cox coefficient of the raw LP) and a proportional-hazards check (Schoenfeld residuals).

Cox models use lifelines (Efron ties, no penalty), as in steps 3 and 4b.

TR: Test setinde kullanılacak ölçütler: SD başına HR, Harrell ve Uno C, Kaplan-Meier, log-rank, kalibrasyon eğimi ve orantılı riskler testi. Hepsi tek bir `evaluate()` fonksiyonunda; birincil analiz ve duyarlılık analizleri aynı kodu kullanır.

In [ ]:
def to_builtin(o):
    """JSON-safe copy: numpy types to Python types; NaN and infinity to None."""
    if isinstance(o, dict):
        return {str(k): to_builtin(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_builtin(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_builtin(o.tolist())
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if np.isfinite(o) else None
    return o

def write_json(obj, path):
    with open(path + '.tmp', 'w') as f:
        json.dump(to_builtin(obj), f, indent=2)
        f.write('\n')
    os.replace(path + '.tmp', path)

def cox_uni(time_, event, x, name):
    """Univariable Cox model (lifelines, Efron ties). Returns the summary dict, the fitter and its data."""
    df = pd.DataFrame({'time': np.asarray(time_, dtype=float), 'event': np.asarray(event, dtype=int),
                       name: np.asarray(x, dtype=float)})
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        cph = CoxPHFitter().fit(df, duration_col='time', event_col='event')
    s = cph.summary.loc[name]
    notes = sorted({f'{w.category.__name__}: {str(w.message)[:160]}' for w in caught
                    if w.category.__module__.startswith('lifelines') or issubclass(w.category, RuntimeWarning)})
    return ({'n': int(len(df)), 'deaths': int(df['event'].sum()), 'coef': float(s['coef']), 'se': float(s['se(coef)']),
             'coef_ci_95': [float(s['coef lower 95%']), float(s['coef upper 95%'])], 'hr': float(s['exp(coef)']),
             'hr_ci_95': [float(s['exp(coef) lower 95%']), float(s['exp(coef) upper 95%'])],
             'p_wald': float(s['p']), 'p_lrt': float(cph.log_likelihood_ratio_test().p_value), 'warnings': notes},
            cph, df)

def uno_tau(time_, tau0, min_frac):
    """Plan §8.2: tau0, unless fewer than min_frac of the patients are still at risk (follow-up >= t) at tau0;
    then the latest time at which at least min_frac are at risk."""
    time_ = np.asarray(time_, dtype=float)
    if np.mean(time_ >= tau0) >= min_frac:
        return float(tau0), f'{tau0:g} months (plan value; {np.mean(time_ >= tau0):.0%} at risk)'
    k = int(np.ceil(min_frac * len(time_)))
    tau = float(np.sort(time_)[::-1][k - 1])
    return tau, f'{tau:.1f} months (fewer than {min_frac:.0%} at risk at {tau0:g} months)'

def uno_c(time_, event, score, tau):
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    return float(concordance_index_ipcw(y, y, np.asarray(score, dtype=float), tau=tau)[0])

def km_rows(time_, event, labels, order, t_os=60.0):
    """n, deaths, median OS and OS at t_os months (95% CIs, lifelines defaults) for each group."""
    rows = []
    for g in order:
        m = labels == g
        row = {'group': g, 'n': int(m.sum()), 'deaths': int(event[m].sum()), 'median_os_months': None,
               'median_ci_95': [None, None], 'os_5y': None, 'os_5y_ci_95': [None, None]}
        if m.sum() > 0:
            kmf = KaplanMeierFitter().fit(time_[m], event[m])
            med = float(kmf.median_survival_time_)
            mci = median_survival_times(kmf.confidence_interval_).iloc[0].to_numpy(dtype=float)
            row['median_os_months'] = med if np.isfinite(med) else None
            row['median_ci_95'] = [float(v) if np.isfinite(v) else None for v in mci]
            if time_[m].max() >= t_os:
                ci = kmf.confidence_interval_
                at = ci.loc[ci.index <= t_os].iloc[-1]
                row['os_5y'] = float(kmf.survival_function_at_times(t_os).iloc[0])
                row['os_5y_ci_95'] = [float(at.iloc[0]), float(at.iloc[1])]
        rows.append(row)
    return rows

def evaluate(time_, event, lp, *, cutoff, tau, boot_idx, z=None, calibration=True, ph_check=True):
    """Metrics of plan §8.2 for one risk score (higher = higher risk) in one analysis population."""
    time_, event, lp = np.asarray(time_, dtype=float), np.asarray(event, dtype=int), np.asarray(lp, dtype=float)
    if z is None:
        z = (lp - lp.mean()) / lp.std(ddof=1)
    out = {'n': int(len(lp)), 'deaths': int(event.sum()), 'lp_mean': float(lp.mean()), 'lp_sd': float(lp.std(ddof=1))}
    out['hr_per_sd'], cph_sd, df_sd = cox_uni(time_, event, z, 'lp_per_sd')
    out['harrell_c'] = harrell_c(time_, event, lp)
    if boot_idx is not None:
        cs = np.array([harrell_c(time_[b], event[b], lp[b]) for b in boot_idx])
        out['harrell_c_ci_95'] = [float(np.percentile(cs, 2.5)), float(np.percentile(cs, 97.5))]
        out['harrell_c_bootstrap'] = {'resamples': int(len(cs)), 'sd': float(cs.std(ddof=1))}
    out['uno_tau_months'] = float(tau)
    out['uno_c'] = uno_c(time_, event, lp, tau)
    high = lp > cutoff
    out['cutoff'] = float(cutoff)
    out['groups'] = km_rows(time_, event, np.where(high, 'high', 'low'), ['high', 'low'])
    out['logrank'], out['hr_high_vs_low'] = None, None
    if 0 < high.sum() < len(high):
        lr = logrank_test(time_[high], time_[~high], event[high], event[~high])
        out['logrank'] = {'chi2': float(lr.test_statistic), 'p': float(lr.p_value)}
        out['hr_high_vs_low'] = cox_uni(time_, event, high.astype(float), 'high_risk')[0]
    if calibration:
        out['calibration_slope'] = cox_uni(time_, event, lp, 'lp_raw')[0]
    if ph_check:
        out['ph_test'] = {}
        for tt in ('rank', 'km'):
            r = proportional_hazard_test(cph_sd, df_sd, time_transform=tt)
            out['ph_test'][tt] = {'statistic': float(r.summary['test_statistic'].iloc[0]),
                                  'p': float(r.summary['p'].iloc[0])}
    return out

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return 'n/a'
    return f'{p:.1e}' if p < 0.001 else f'{p:.3f}'

def fmt_hr(r):
    return 'n/a' if r is None else f"{r['hr']:.2f} ({r['hr_ci_95'][0]:.2f}–{r['hr_ci_95'][1]:.2f}), p = {fmt_p(r['p_wald'])}"

def fmt_c(est, ci=None):
    return f'{est:.3f}' + ('' if ci is None else f' ({ci[0]:.3f}–{ci[1]:.3f})')

def fmt_os(row):
    if row['os_5y'] is None:
        return 'n/a'
    lo, hi = row['os_5y_ci_95']
    return f"{100 * row['os_5y']:.1f}% ({100 * lo:.1f}–{100 * hi:.1f})"

def fmt_median(row):
    m, (lo, hi) = row['median_os_months'], row['median_ci_95']
    if m is None:
        return 'not reached'
    return f"{m:.1f} mo ({'?' if lo is None else f'{lo:.1f}'}–{'no upper limit' if hi is None else f'{hi:.1f}'})"

print('validation functions defined')

## 6. Settings, seeds and checkpoints
All settings come from the locked plan, and the seed formulas written in the plan are compared as text, so the code cannot drift from them. `cached()` stores each finished unit in `results/05/checkpoints_05/` with a fingerprint of the code version, plan, data and package versions; a stored unit is reused only if its fingerprint matches.

TR: Ayarlar kilitli plandan okunur; tohum formülleri planla metin olarak karşılaştırılır. Biten her iş Drive'a kaydedilir; aynı kod, plan, veri ve paket sürümleriyle yeniden çalıştırınca yeniden hesaplanmaz.

In [ ]:
SEED = PLAN['master_seed']
CFG = {'l1_ratio': PLAN['model']['l1_ratio'], 'n_alphas': PLAN['model']['n_alphas'],
       'alpha_min_ratio': PLAN['model']['alpha_min_ratio'], 'cv_folds': PLAN['model']['cv_folds'],
       'top_n': PLAN['features']['variance_filter_top_n'], 'ddof': PLAN['features']['variance_ddof'],
       'max_iter': 100000, 'tol': 1e-7}               # scikit-survival defaults, as in step 4b
assert PLAN['model']['ties'] == 'breslow' and PLAN['model']['lambda_rule_primary'] == 'lambda_min'
assert PLAN['model']['implementation'] == 'scikit-survival CoxnetSurvivalAnalysis'
assert PLAN['model']['cv_stratified_by'] == EVENT

S, NC, IV = PLAN['seeds'], PLAN['step5_negative_control'], PLAN['step5_internal_validation']
LK = NC['leakage_check']
assert S['final_model_cv_folds'] == SEED and S['validation_bootstrap'] == SEED + 400
assert S['permutation_labels'] == 'SEED + 1000 + i (i = 0..99)' and S['permutation_cv_folds'] == 'SEED + 2000 + i'
assert S['leakage_check_labels'] == 'SEED + 3000 + i (i = 0..9)' and S['leakage_check_outer_split'] == 'SEED + 3100 + i'
assert S['leakage_check_inner_folds'] == 'SEED + 3200 + 10*i + j'
assert NC['permutation'] == '(time, event) pairs permuted jointly among the primary training patients'
assert NC['evaluation'] == 'C-index on the true test labels' and LK['nested_cv'] == '5-fold x 1 repeat'
assert IV['test_scaling'] == 'training means and SDs'

N_PERM, N_LEAK, LEAK_FOLDS, LEAK_STOP = NC['n_permutations'], LK['n_permutations'], 5, LK['stop_if_mean_c_above']
N_BOOT = IV['bootstrap_resamples']
TAU0, MIN_AT_RISK = PLAN['endpoints']['uno_c_tau_months'], PLAN['endpoints']['uno_c_min_at_risk_fraction']
NESTED_C = M4B['nested_cv']['min']['mean_c']            # reference for the third red flag of plan §13

def seed_perm_labels(i): return SEED + 1000 + i
def seed_perm_cv(i): return SEED + 2000 + i
def seed_leak_labels(i): return SEED + 3000 + i
def seed_leak_outer(i): return SEED + 3100 + i
def seed_leak_inner(i, j): return SEED + 3200 + 10 * i + j

def permuted(seed):
    """(time, event) pairs permuted jointly among the primary training patients (plan §9)."""
    perm = np.random.default_rng(seed).permutation(len(T))
    return T[perm], E[perm], perm

FP_BASE = {'code': CODE_VERSION, 'plan_json_sha256': LOCK['plan_json_sha256'], 'cfg': CFG,
           'inputs': {k: INPUT_SHA[k]['sha256'] for k in ('clinical_all', 'split', 'qc_samples', 'expr_train')},
           'packages': {k: VERSIONS[k] for k in ('numpy', 'scipy', 'scikit-learn', 'scikit-survival')}}

def cached(unit, params, compute):
    """Stored result if computed with the same code, plan, data, packages and parameters; otherwise compute and store."""
    path = f'{CKPT}/{unit}.pkl'
    fp = hashlib.sha256(json.dumps({**FP_BASE, 'unit': unit, 'params': params}, sort_keys=True,
                                   default=str).encode()).hexdigest()
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            if obj.get('fingerprint') == fp:
                return obj['result'], True
        except Exception:
            pass
    result = compute()
    with open(path + '.tmp', 'wb') as f:
        pickle.dump({'fingerprint': fp, 'result': result}, f, protocol=4)
    os.replace(path + '.tmp', path)
    return result, False

def procedure(Xs, Ts, Es, cv_seed):
    return fit_procedure(Xs, Ts, Es, CFG['l1_ratio'], CFG['top_n'], cv_seed, CFG, N_JOBS, JOBLIB_TMP)

def fmt_min(sec):
    return f'{sec / 60:.0f} min' if sec >= 90 else f'{sec:.0f} s'

class Progress:
    """Units done and an estimate of the time left, from the units computed in this session."""
    def __init__(self, total):
        self.total, self.done, self.secs = total, 0, []
    def step(self, seconds):
        self.done += 1
        if seconds is not None:
            self.secs.append(seconds)
    def eta(self):
        if self.done == self.total:
            return 'done'
        if not self.secs:
            return 'loaded from checkpoints so far'
        return f'about {fmt_min((self.total - self.done) * float(np.mean(self.secs)))} left'

# Figure style (as in step 4b): two validated series colours, neutral ink for text, light solid grid.
BLUE, ORANGE, GREY = '#2a78d6', '#eb6834', '#b9b8b3'
TERT_COLORS = ['#86b6ef', '#2a78d6', '#104281']           # validated ordinal ramp: T1 light ... T3 dark
INK, INK2, MUTED, GRID = '#0b0b0b', '#52514e', '#898781', '#e6e5e1'
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 9, 'axes.edgecolor': INK2,
                     'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2, 'axes.grid': True,
                     'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False, 'legend.labelcolor': INK2})

def savefig(fig, name):
    fig.savefig(f'{FIG}/{name}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

n_ckpt = len([f for f in os.listdir(CKPT) if f.endswith('.pkl')])
print(f'settings loaded: {N_LEAK} leakage-check permutations x {LEAK_FOLDS} folds, {N_PERM} null models, '
      f'{N_BOOT} bootstrap resamples; {n_ckpt} checkpoint files already in results/05/checkpoints_05')

## 7. Reproducibility check: the step-4b fit, run again (training set only)
The whole procedure runs once more on the 373 training patients with the plan's CV seed. The result must match the summary of the frozen model recorded in `manifest_04b.json`: number of genes, penalty, position on the grid, and no path events. The model file is not opened here; its genes and coefficients are compared in section 9, after its hash has been checked. If this check fails, this runtime does not reproduce step 4b, and the permuted models would not be fitted with the same procedure, so the notebook stops.

TR: 4b'deki final model, bütün yöntemle eğitim verisinden yeniden kurulur ve 4b manifestosundaki özetle (gen sayısı, λ, ızgaradaki yer) karşılaştırılır. Bu, bu oturumdaki kodun ve paketlerin 4b ile aynı sonucu verdiğini gösterir; permütasyon modellerinin aynı yöntemle kurulacağının güvencesidir. Model dosyası burada açılmaz. Yaklaşık 1 dakika. Eşleşmezse not defteri durur.

In [ ]:
def reproduce_unit():
    t0 = time.time()
    proc = procedure(X, T, E, S['final_model_cv_folds'])
    m = model_at(proc, proc['idx_min'])
    return {'entrez': GENE_STR[m['gene_pos']].tolist(), 'coef': m['coef'].tolist(), 'lambda': m['lambda'],
            'position': m['idx'] + 1, 'n_valid': int(proc['n_valid']),
            'at_boundary': bool(proc['idx_min'] == proc['n_valid'] - 1),
            'n_genes_1se': int((proc['coef'][:, proc['idx_1se']] != 0).sum()), 'position_1se': proc['idx_1se'] + 1,
            'path_events': path_events(proc), 'seconds': time.time() - t0}

t_sec = time.time()
REPRO, _hit = cached('reproduce_4b_fit', {'cv_seed': S['final_model_cv_folds']}, reproduce_unit)
F4B, FF4B = M4B['frozen_model'], M4B['final_fit']
REPRO_VS_MANIFEST = {
    'number of genes': (len(REPRO['entrez']), F4B['n_genes']),
    'lambda_min': (REPRO['lambda'], F4B['lambda']),
    'grid position of lambda_min': (REPRO['position'], FF4B['lambda_min_position']),
    'valid grid size': (REPRO['n_valid'], FF4B['valid_grid_size']),
    'genes at lambda_1se': (REPRO['n_genes_1se'], FF4B['n_genes_lambda_1se']),
    'path events': (REPRO['path_events'], FF4B['path_events']),
}
_ok = {}
for name, (mine, ref) in REPRO_VS_MANIFEST.items():
    _ok[name] = bool(np.isclose(mine, ref, rtol=1e-9, atol=0)) if isinstance(ref, float) else mine == ref
    _show = (lambda d: f'{sum(d.values())} in total') if isinstance(ref, dict) else (lambda v: v)
    print(f"  {'match' if _ok[name] else 'DIFFERENT':9s} {name}: {_show(mine)} (step 4b: {_show(ref)})")
REPRO_OK = all(_ok.values())
RUNTIME['reproducibility_check_s'] = round(time.time() - t_sec, 1)
if not REPRO_OK:
    _env = {k: (VERSIONS[k], M4B['environment'].get(k)) for k in ('numpy', 'scipy', 'scikit-learn', 'scikit-survival')}
    raise RuntimeError('STOP: this runtime does not reproduce the step-4b fit, so the permuted models would not be '
                       f'fitted with the same procedure. Package versions (now, step 4b): {_env}. '
                       'Send this message to Claude; the test set was not opened.')
print(f"Step-4b fit reproduced: {len(REPRO['entrez'])} genes at lambda = {REPRO['lambda']:.6g} "
      f"(grid position {REPRO['position']}).{' Loaded from checkpoint.' if _hit else ''}")
if not _hit:
    UNIT_S = REPRO['seconds']
    print(f'One full fit took {fmt_min(UNIT_S)}. Rough time for the permutations: leakage check about '
          f'{fmt_min(0.8 * UNIT_S * N_LEAK * LEAK_FOLDS)}, null models about {fmt_min(UNIT_S * N_PERM)}.')

## 8. Leakage check (training set only) — plan §9.2, gate 1
The survival labels of the 373 training patients are permuted (time and event together), which breaks every real link with expression. On each of 10 permuted sets, nested CV (5 outer folds × 1 repeat) runs the whole procedure inside each outer training fold, exactly as in step 4b, and Harrell's C is computed on each outer test fold. Without leakage the mean C is about 0.5. **If it is above 0.55, the notebook stops here and the test set stays closed.**

TR: Yalnızca eğitim verisi. Sağkalım etiketleri (süre ve olay birlikte) hastalar arasında karıştırılır; böylece ifade ile sağkalım arasındaki bütün gerçek bağ kopar. 10 karıştırılmış veri setinin her birinde iç içe CV çalışır. Gerçek sinyal olmadığı için ortalama C 0,5 civarında çıkmalı. 0,55'in üstündeyse kodda bir sızıntı var demektir: not defteri durur ve test seti hiç açılmaz. Yaklaşık 15–25 dakika.

In [ ]:
def leak_unit(i, j, otr, ote, Tp, Ep):
    t0 = time.time()
    proc = procedure(X[otr], Tp[otr], Ep[otr], seed_leak_inner(i, j))
    m = model_at(proc, proc['idx_min'])
    lp = lp_of(m, X[ote])                       # outer test patients z-scored with the outer training statistics
    return {'perm': i, 'fold': j, 'label_seed': seed_leak_labels(i), 'outer_seed': seed_leak_outer(i),
            'inner_seed': seed_leak_inner(i, j), 'n_train': int(len(otr)), 'n_test': int(len(ote)),
            'deaths_test': int(Ep[ote].sum()), 'n_genes': int(len(m['coef'])), 'lambda': m['lambda'],
            'position': m['idx'] + 1, 'n_valid': int(proc['n_valid']),
            'at_boundary': bool(proc['idx_min'] == proc['n_valid'] - 1),
            'c': harrell_c(Tp[ote], Ep[ote], lp), **path_events(proc), 'seconds': time.time() - t0,
            'computed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}

t_sec = time.time()
LEAK_ROWS, prog = [], Progress(N_LEAK * LEAK_FOLDS)
for i in range(N_LEAK):
    Tp, Ep, _perm = permuted(seed_leak_labels(i))
    folds = list(StratifiedKFold(n_splits=LEAK_FOLDS, shuffle=True, random_state=seed_leak_outer(i))
                 .split(np.zeros(len(Ep)), Ep))
    for j, (otr, ote) in enumerate(folds):
        params = {'perm': i, 'fold': j, 'label_seed': seed_leak_labels(i), 'outer_seed': seed_leak_outer(i),
                  'inner_seed': seed_leak_inner(i, j), 'outer_folds': LEAK_FOLDS}
        res, hit = cached(f'leak_p{i}_f{j}', params, lambda: leak_unit(i, j, otr, ote, Tp, Ep))
        LEAK_ROWS.append(res)
        prog.step(None if hit else res['seconds'])
    _cs = [r['c'] for r in LEAK_ROWS[-LEAK_FOLDS:]]
    print(f"permuted set {i + 1:2d}/{N_LEAK}: mean C {np.mean(_cs):.3f} (folds {min(_cs):.3f}–{max(_cs):.3f}); "
          f"genes per fold {[r['n_genes'] for r in LEAK_ROWS[-LEAK_FOLDS:]]}; {prog.eta()}")

LEAK = pd.DataFrame(LEAK_ROWS)
LEAK_MEAN_C = float(LEAK['c'].mean())
LEAKAGE_PASSED = bool(LEAK_MEAN_C <= LEAK_STOP)
_pm = LEAK.groupby('perm')['c'].mean()
LEAK_SUMMARY = {
    'n_permutations': N_LEAK, 'outer_folds': LEAK_FOLDS, 'units': int(len(LEAK)), 'mean_c': LEAK_MEAN_C,
    'sd_c_folds': float(LEAK['c'].std(ddof=1)), 'min_c': float(LEAK['c'].min()), 'max_c': float(LEAK['c'].max()),
    'mean_c_per_permutation': [float(v) for v in _pm], 'share_empty_models': float((LEAK['n_genes'] == 0).mean()),
    'median_genes': float(LEAK['n_genes'].median()), 'stop_if_mean_c_above': LEAK_STOP, 'passed': LEAKAGE_PASSED,
    'path_events': {k: int(LEAK[k].sum()) for k in ('spurious_stop_restarts', 'genuine_early_stops',
                                                     'numerical_failures', 'convergence_warnings', 'penalties_dropped')},
    'folds_lambda_min_at_boundary': int(LEAK['at_boundary'].sum())}
LEAK.to_csv(f'{RES}/leakage_check_05.csv', index=False)
write_json(LEAK_SUMMARY, f'{RES}/leakage_check_05.json')
RUNTIME['leakage_check_s'] = round(time.time() - t_sec, 1)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
for i in range(N_LEAK):
    cs = LEAK.loc[LEAK['perm'] == i, 'c'].to_numpy()
    ax.scatter(i + 1 + np.linspace(-0.18, 0.18, len(cs)), cs, s=24, color=BLUE, edgecolor='white', linewidth=0.8, zorder=3)
    ax.plot([i + 0.72, i + 1.28], [cs.mean()] * 2, color=INK, lw=1.4, zorder=4)
for y, txt, col in ((0.5, 'no information (0.5)', MUTED), (LEAK_STOP, f'stop rule ({LEAK_STOP})', INK2)):
    ax.axhline(y, color=col, lw=1.0, ls='--', zorder=2)
    ax.text(N_LEAK + 0.55, y, txt, color=col, fontsize=8, va='center', ha='left')
ax.set_xlim(0.4, N_LEAK + 0.5)
ax.set_xticks(range(1, N_LEAK + 1))
ax.set_xlabel('Permuted training set (dots: outer test folds; bar: mean)')
ax.set_ylabel("Harrell's C")
ax.set_title(f'Leakage check: mean C {LEAK_MEAN_C:.3f} over {len(LEAK)} outer folds', loc='left', fontsize=10, color=INK)
savefig(fig, 'leakage_check_05.png')

if not LEAKAGE_PASSED:
    raise RuntimeError(f'STOP (plan §9.2): the mean C of the permuted-label nested CV is {LEAK_MEAN_C:.3f} > {LEAK_STOP}. '
                       'The test set was not opened. Search the pipeline for leakage before going further.')
print(f"Gate 1 passed: mean C {LEAK_MEAN_C:.3f} <= {LEAK_STOP} (folds {LEAK_SUMMARY['min_c']:.3f}–{LEAK_SUMMARY['max_c']:.3f}; "
      f"empty models {LEAK_SUMMARY['share_empty_models']:.0%}).")

## 9. Frozen model check — plan §8, gate 2
1. The SHA-256 of `results/04/model_04.json` must equal the value in `manifest_04b.json` (twice: `frozen_model` and `outputs`) and the value recorded in `progress_step4b.md`.
2. Its contents must agree with the plan, the manifest and the training population.
3. The training means and SDs stored for each gene, the risk cutoff (median training LP), the tertile cutoffs and the LP mean and SD are recomputed from the training data.
4. The model re-fitted in section 7 must have exactly the same genes, and the same coefficients (relative tolerance 10⁻⁶, as in step 4b).

TR: Dondurulmuş modelin SHA-256 özeti, 4b manifestosundaki iki kayıtla ve progress_step4b.md'deki değerle karşılaştırılır. Sonra içeriği kontrol edilir: eğitim ortalamaları ve SD'leri, risk eşiği (eğitim medyanı) ve tertil eşikleri eğitim verisinden yeniden hesaplanır; 7. bölümde yeniden kurulan modelin genleri ve katsayıları dosyadakilerle birebir aynı olmalı. Herhangi bir fark varsa not defteri durur.

In [ ]:
assert globals().get('LEAKAGE_PASSED') is True, 'Gate 1 (leakage check) has not passed in this session. Run all cells in order.'
MODEL_PATH = f'{RES4}/model_04.json'
assert os.path.exists(MODEL_PATH), 'results/04/model_04.json not found.'
MODEL_SHA = sha256_file(MODEL_PATH)
HASH_REFS = {'manifest_04b.json, frozen_model.sha256': M4B['frozen_model']['sha256'],
             'manifest_04b.json, outputs entry': M4B['outputs'].get('results/04/model_04.json'),
             'progress_step4b.md': EXPECTED_MODEL_SHA256}
print(f'model_04.json SHA-256: {MODEL_SHA}')
for name, ref in HASH_REFS.items():
    print(f"  {'match' if ref == MODEL_SHA else 'DIFFERENT':9s} {name} ({str(ref)[:12]}…)")
assert all(ref == MODEL_SHA for ref in HASH_REFS.values()), \
    'STOP: the model file differs from the frozen model recorded after step 4b. The test set stays closed.'

with open(MODEL_PATH) as f:
    MODEL = json.load(f)
if MODEL['null_model']:
    raise RuntimeError('The frozen model is empty (null result in step 4b); H1 cannot be tested. Report the null result.')
_genes = MODEL['genes']
_ent = [str(g['entrez']) for g in _genes]
_pos_of = {g: k for k, g in enumerate(GENE_STR)}
CONTENT_OK = {
    'plan JSON hash in the model file': MODEL['plan_json_sha256'] == LOCK['plan_json_sha256'],
    'number of genes (file, list, manifest)': MODEL['n_genes'] == len(_genes) == len(set(_ent)) == M4B['frozen_model']['n_genes'],
    'penalty (file vs manifest)': bool(np.isclose(MODEL['penalty']['lambda'], M4B['frozen_model']['lambda'], rtol=1e-12, atol=0)),
    f'training population ({len(T)} patients, {E.sum()} deaths)': (MODEL['training_population']['patients'] == len(T)
                                                                    and MODEL['training_population']['deaths'] == int(E.sum())),
    'every model gene is in the expression data': all(g in _pos_of for g in _ent),
}
for name, ok in CONTENT_OK.items():
    print(f"  {'ok' if ok else 'FAILED':9s} {name}")
assert all(CONTENT_OK.values()), 'STOP: model_04.json content check failed.'

_order = np.argsort([_pos_of[g] for g in _ent])          # column order, as model_at() returned it in step 4b
FROZEN = {'gene_pos': np.array([_pos_of[_ent[k]] for k in _order]),
          'coef': np.array([_genes[k]['coef'] for k in _order], dtype=float),
          'mean': np.array([_genes[k]['train_mean'] for k in _order], dtype=float),
          'sd': np.array([_genes[k]['train_sd'] for k in _order], dtype=float),
          'lambda': float(MODEL['penalty']['lambda']), 'idx': int(MODEL['penalty']['grid_position']) - 1}
FROZEN_ENTREZ = GENE_STR[FROZEN['gene_pos']]
LP_TRAIN = lp_of(FROZEN, X)
RECOMPUTED = {
    'gene means (training)': (X[:, FROZEN['gene_pos']].mean(axis=0), FROZEN['mean']),
    'gene SDs (training)': (X[:, FROZEN['gene_pos']].std(axis=0, ddof=CFG['ddof']), FROZEN['sd']),
    'risk cutoff (median training LP)': (np.median(LP_TRAIN), MODEL['risk_groups']['cutoff']),
    'tertile cutoffs (training LP)': (np.quantile(LP_TRAIN, [1 / 3, 2 / 3]), MODEL['risk_groups']['tertile_cutoffs_descriptive']),
    'training LP mean': (LP_TRAIN.mean(), MODEL['training_lp']['mean']),
    'training LP SD': (LP_TRAIN.std(ddof=1), MODEL['training_lp']['sd']),
}
STAT_CHECK = {}
for name, (a, b) in RECOMPUTED.items():
    a, b = np.atleast_1d(np.asarray(a, dtype=float)), np.atleast_1d(np.asarray(b, dtype=float))
    STAT_CHECK[name] = {'max_abs_diff': float(np.max(np.abs(a - b))), 'ok': bool(np.allclose(a, b, rtol=1e-9, atol=1e-12))}
    print(f"  {'ok' if STAT_CHECK[name]['ok'] else 'FAILED':9s} {name}: max |difference| {STAT_CHECK[name]['max_abs_diff']:.1e}")
assert all(v['ok'] for v in STAT_CHECK.values()), 'STOP: training statistics in model_04.json do not match the training data.'

_cf, _cr = dict(zip(FROZEN_ENTREZ, FROZEN['coef'])), dict(zip(REPRO['entrez'], REPRO['coef']))
_same_genes = set(_cf) == set(_cr)
REPRO_CHECK = {
    'same genes': _same_genes,
    'same coefficients (rtol 1e-6, atol 1e-9)': _same_genes and all(
        np.isclose(_cr[g], _cf[g], rtol=1e-6, atol=1e-9) for g in _cf),
    'same penalty (rtol 1e-9)': bool(np.isclose(REPRO['lambda'], FROZEN['lambda'], rtol=1e-9, atol=0)),
    'same grid position': REPRO['position'] == MODEL['penalty']['grid_position'],
}
MAX_DCOEF = max(abs(_cr[g] - _cf[g]) for g in _cf) if _same_genes else float('nan')
for name, ok in REPRO_CHECK.items():
    print(f"  {'ok' if ok else 'FAILED':9s} re-fit vs file: {name}")
assert all(REPRO_CHECK.values()), 'STOP: the model re-fitted from the training data differs from model_04.json.'

MODEL_VERIFIED = True
CUTOFF = float(MODEL['risk_groups']['cutoff'])
TERT = [float(v) for v in MODEL['risk_groups']['tertile_cutoffs_descriptive']]
write_json({'model_file': 'results/04/model_04.json', 'sha256': MODEL_SHA, 'hash_references': HASH_REFS,
            'content_checks': CONTENT_OK, 'recomputed_from_training_data': STAT_CHECK,
            'refit_vs_file': REPRO_CHECK, 'refit_max_abs_coef_difference': MAX_DCOEF,
            'refit_vs_manifest_04b': {k: {'now': v[0], 'step4b': v[1]} for k, v in REPRO_VS_MANIFEST.items()},
            'n_genes': int(len(FROZEN['coef'])), 'lambda': FROZEN['lambda'], 'cutoff': CUTOFF,
            'tertile_cutoffs': TERT, 'verified': True}, f'{RES}/frozen_model_check_05.json')
print(f"Gate 2 passed: frozen model verified ({len(FROZEN['coef'])} genes, lambda {FROZEN['lambda']:.6g}, "
      f'training cutoff {CUTOFF:.4f}); re-fit identical (max |coefficient difference| {MAX_DCOEF:.1e}).')

## 10. Null models for the negative control (training set only) — plan §9.1
On each of 100 permuted training sets (time and event permuted together), the whole procedure runs again: variance filter, z-score, penalty grid, 10-fold CV and λ_min. Only the selected genes, coefficients and z-score statistics are kept. All 100 models are fixed and saved before the test set is opened; they are scored in section 13.

TR: Negatif kontrolün modelleri. Etiketler 100 kez karıştırılır ve her seferinde bütün yöntem baştan çalışır (varyans filtresi, z-skor, ceza ızgarası, 10 katlı CV, λ_min). Bu modeller test seti açılmadan önce kurulup Drive'a kaydedilir; puanlamaları 13. bölümde yapılır. En uzun kısım: yaklaşık 35–50 dakika. Bağlantı koparsa Tümünü çalıştır'a bas; kaldığı yerden devam eder.

In [ ]:
assert globals().get('MODEL_VERIFIED') is True, 'Gate 2 (frozen model check) has not passed in this session. Run all cells in order.'

def null_unit(i):
    t0 = time.time()
    Tp, Ep, perm = permuted(seed_perm_labels(i))
    proc = procedure(X, Tp, Ep, seed_perm_cv(i))
    m = model_at(proc, proc['idx_min'])
    return {'perm': i, 'label_seed': seed_perm_labels(i), 'cv_seed': seed_perm_cv(i),
            'perm_sha256': hashlib.sha256(perm.astype(np.int64).tobytes()).hexdigest()[:16],
            'gene_pos': m['gene_pos'], 'coef': m['coef'], 'mean': m['mean'], 'sd': m['sd'],
            'lambda': m['lambda'], 'position': m['idx'] + 1, 'n_genes': int(len(m['coef'])),
            'n_valid': int(proc['n_valid']), 'at_boundary': bool(proc['idx_min'] == proc['n_valid'] - 1),
            **path_events(proc), 'seconds': time.time() - t0,
            'computed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}

t_sec = time.time()
NULL_MODELS, prog = [], Progress(N_PERM)
for i in range(N_PERM):
    params = {'perm': i, 'label_seed': seed_perm_labels(i), 'cv_seed': seed_perm_cv(i)}
    res, hit = cached(f'null_p{i:03d}', params, lambda: null_unit(i))
    NULL_MODELS.append(res)
    prog.step(None if hit else res['seconds'])
    if (i + 1) % 10 == 0 or i + 1 == N_PERM:
        print(f"{i + 1:3d}/{N_PERM} null models; empty so far: {sum(r['n_genes'] == 0 for r in NULL_MODELS)}; {prog.eta()}")

NULL_DONE = len(NULL_MODELS) == N_PERM
NULL_DONE_UTC = max(r['computed_utc'] for r in NULL_MODELS)      # when the last null model was computed
_skip = ('gene_pos', 'coef', 'mean', 'sd')
NULL_TRAIN = pd.DataFrame([{k: v for k, v in r.items() if k not in _skip} for r in NULL_MODELS])
NULL_TRAIN['genes'] = [' '.join(GENE_STR[r['gene_pos']]) for r in NULL_MODELS]
NULL_TRAIN.to_csv(f'{RES}/null_models_05.csv', index=False)
RUNTIME['null_models_s'] = round(time.time() - t_sec, 1)
_ne = NULL_TRAIN['n_genes']
print(f'All {N_PERM} null models fixed before the test set is opened (last one computed {NULL_DONE_UTC}): empty {int((_ne == 0).sum())}, '
      f'median genes in non-empty models {(_ne[_ne > 0].median() if (_ne > 0).any() else 0):.0f}, '
      f"numerical failures {int(NULL_TRAIN['numerical_failures'].sum())}.")

## 11. Open the test set — gate 3
The three gates must have passed in this session. The model file and the test expression file are hashed again right before opening. The test rows of the clinical and QC tables and the test expression file are loaded here for the first time, and the QC rule of plan §2 is applied (2 exclusions expected). Every opening is appended to `results/05/test_access_log_05.jsonl` with a timestamp.

TR: Test seti ilk kez burada açılır. Önce üç kapının (sızıntı kontrolü, model kontrolü, permütasyon modelleri) bu oturumda geçtiği kontrol edilir; model ve test ifade dosyası yeniden özetlenir. QC kuralı uygulanır (2 örnek çıkar). Her açılış `test_access_log_05.jsonl` dosyasına tarih ve saatle yazılır.

In [ ]:
GATES = {'gate 1: leakage check passed (section 8)': globals().get('LEAKAGE_PASSED') is True,
         'gate 2: frozen model verified (section 9)': globals().get('MODEL_VERIFIED') is True,
         f'all {N_PERM} null models fixed (section 10)': globals().get('NULL_DONE') is True}
for name, ok in GATES.items():
    print(f"  {'passed' if ok else 'NOT PASSED':10s} {name}")
assert all(GATES.values()), 'A gate has not passed in this session. Run all cells from the top, in order.'
assert sha256_file(MODEL_PATH) == MODEL_SHA == EXPECTED_MODEL_SHA256, 'model_04.json changed during this session. Stop.'
REL_TEST = LOCK['input_files']['expr_test']
assert sha256_file(f'{PROJECT_DIR}/{REL_TEST}') == LOCK['input_sha256']['expr_test'], 'The test expression file changed. Stop.'

TEST_OPENED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
ACCESS_LOG = f'{RES}/test_access_log_05.jsonl'
N_PREVIOUS_OPENINGS = sum(1 for _ in open(ACCESS_LOG)) if os.path.exists(ACCESS_LOG) else 0
with open(ACCESS_LOG, 'a') as f:
    f.write(json.dumps({'utc': TEST_OPENED_UTC, 'event': 'test set opened', 'code_version': CODE_VERSION,
                        'model_sha256': MODEL_SHA, 'leakage_check_mean_c': round(LEAK_MEAN_C, 6),
                        'null_models_fixed_utc': NULL_DONE_UTC}) + '\n')

clin_all = pd.read_csv(f'{CLEAN}/clinical_clean_all.csv')
te = clin_all[clin_all['sample_id'].isin(test_ids)].copy()
del clin_all                                        # training and external rows leave memory here
qc_all = pd.read_csv(f'{PROC}/qc_samples.csv')
qc_te = qc_all.loc[qc_all['sample_id'].isin(test_ids), ['sample_id', FLAG_P, FLAG_S]].copy()
del qc_all
assert qc_te['sample_id'].is_unique and set(qc_te['sample_id']) == test_ids, 'QC rows do not match the test set'
qc_te[FLAG_P], qc_te[FLAG_S] = as_bool(qc_te[FLAG_P]), as_bool(qc_te[FLAG_S])
assert te['sample_id'].is_unique and len(te) == TST['patients'] and int(te[EVENT].sum()) == TST['deaths'], \
    'test counts differ from the plan'
te = te.merge(qc_te, on='sample_id', how='left', validate='one_to_one')
EXCLUDED_TE = sorted(te.loc[te[FLAG_P], 'sample_id'])
_exp = PLAN['qc']['expected_exclusions']['internal_test']
assert len(EXCLUDED_TE) == _exp, f'{len(EXCLUDED_TE)} test samples are flagged {FLAG_P}; the plan expects {_exp}.'
tpop = te.loc[~te[FLAG_P]].sort_values('sample_id').reset_index(drop=True)
T_TE = tpop[TIME].to_numpy(dtype=float)
E_TE = tpop[EVENT].to_numpy(dtype=int)
CORR_TE = tpop[FLAG_S].to_numpy(dtype=bool)
SID_TE = tpop['sample_id'].to_numpy()
assert np.isfinite(T_TE).all() and (T_TE > 0).all() and set(np.unique(E_TE)) <= {0, 1}

expr_te = pd.read_parquet(f'{PROJECT_DIR}/{REL_TEST}')
if 'gene' in expr_te.columns:
    expr_te = expr_te.set_index('gene')
assert set(expr_te.columns) == test_ids, 'test expression columns are not exactly the test samples'
expr_te.index = [gene_id_text(g) for g in expr_te.index]
assert expr_te.index.is_unique and set(expr_te.index) == set(GENE_STR), 'the test file does not have the training genes'
X_TE = expr_te.loc[list(GENE_STR), list(SID_TE)].T.to_numpy(dtype=np.float64)    # same gene order as training
del expr_te
assert np.isfinite(X_TE).all(), 'missing or infinite test expression values'
print(f'TEST SET OPENED ({TEST_OPENED_UTC}; earlier openings in the log: {N_PREVIOUS_OPENINGS}).')
print(f"Test split: {len(te)} patients, {int(te[EVENT].sum())} deaths; excluded by {FLAG_P}: {EXCLUDED_TE}.")
FU_TE = float(KaplanMeierFitter().fit(T_TE, 1 - E_TE).median_survival_time_)      # reverse Kaplan-Meier
print(f'Primary test population: {len(T_TE)} patients, {E_TE.sum()} deaths; correlation outliers kept: {CORR_TE.sum()}; '
      f'median follow-up (reverse KM) {FU_TE:.1f} months.')

## 12. Primary analysis — H1 and the metrics of plan §8.2
Test expression is z-scored with the training means and SDs stored in the frozen model (plan §8.1), and LP = Σ coef · z. **H1: the Cox HR per 1 SD of the test LP is above 1 and its 95% CI excludes 1.** Also: Harrell's C with a 1,000-resample percentile bootstrap CI (seed 20261415), Uno's C, Kaplan–Meier curves at the numeric training cutoff with the log-rank test and HR high vs low, the calibration slope and a proportional-hazards check. Tertiles (training tertile cutoffs) are descriptive only.

TR: Asıl sonuç. Test ifadesi eğitim ortalama ve SD'leriyle ölçeklenir ve risk skoru (LP) hesaplanır. H1: SD başına HR > 1 ve %95 GA 1'i dışlıyor. Ayrıca Harrell C (bootstrap GA), Uno C, eğitim eşiğiyle Kaplan-Meier + log-rank, kalibrasyon eğimi (1 ideal; 1'in altı aşırı öğrenme işareti) ve orantılı riskler testi. Tertiller yalnızca betimleyici. Son grafik eğitim ve test LP dağılımlarını karşılaştırır; eğitimdeki dağılımın daha geniş olması beklenir, çünkü model o hastalara uydurulmuştu.

In [ ]:
t_sec = time.time()
LP_TE = lp_of(FROZEN, X_TE)
TAU, TAU_RULE = uno_tau(T_TE, TAU0, MIN_AT_RISK)
BOOT_IDX = np.random.default_rng(S['validation_bootstrap']).integers(0, len(T_TE), size=(N_BOOT, len(T_TE)))
PRIMARY = evaluate(T_TE, E_TE, LP_TE, cutoff=CUTOFF, tau=TAU, boot_idx=BOOT_IDX)
PRIMARY['uno_tau_rule'] = TAU_RULE
H = PRIMARY['hr_per_sd']
H1_MET = bool(H['hr'] > 1 and H['hr_ci_95'][0] > 1)
LR_P = PRIMARY['logrank']['p'] if PRIMARY['logrank'] else None          # None only if one risk group is empty
HRG = PRIMARY['hr_high_vs_low'] or {'hr': None, 'hr_ci_95': [None, None]}
TERT_LABELS = ['T1 (low)', 'T2', 'T3 (high)']
TERT_TE = np.where(LP_TE <= TERT[0], TERT_LABELS[0], np.where(LP_TE <= TERT[1], TERT_LABELS[1], TERT_LABELS[2]))
TERTILES = km_rows(T_TE, E_TE, TERT_TE, TERT_LABELS)
RUNTIME['primary_analysis_s'] = round(time.time() - t_sec, 1)

print(f"Primary test population: n = {PRIMARY['n']}, deaths = {PRIMARY['deaths']}")
print(f"H1  HR per SD of LP: {fmt_hr(H)}  ->  {'MET' if H1_MET else 'NOT MET'}")
print(f"Harrell C: {fmt_c(PRIMARY['harrell_c'], PRIMARY['harrell_c_ci_95'])}; "
      f"Uno C: {PRIMARY['uno_c']:.3f} (tau = {TAU_RULE})")
for row in PRIMARY['groups']:
    print(f"  {row['group']:5s} risk: n = {row['n']:3d}, deaths = {row['deaths']:3d}, median OS {fmt_median(row)}, "
          f'5-year OS {fmt_os(row)}')
if PRIMARY['logrank']:
    print(f"Log-rank (training cutoff {CUTOFF:.4f}): chi2 = {PRIMARY['logrank']['chi2']:.2f}, p = {fmt_p(LR_P)}; "
          f"HR high vs low: {fmt_hr(PRIMARY['hr_high_vs_low'])}")
else:
    print(f'Log-rank not computable: every test patient is on one side of the training cutoff {CUTOFF:.4f}.')
cs_ = PRIMARY['calibration_slope']
print(f"Calibration slope: {cs_['coef']:.2f} ({cs_['coef_ci_95'][0]:.2f}–{cs_['coef_ci_95'][1]:.2f}); 1 = ideal, below 1 = overfitting")
print(f"Proportional hazards (Schoenfeld, descriptive): p = {fmt_p(PRIMARY['ph_test']['rank']['p'])} (rank time), "
      f"{fmt_p(PRIMARY['ph_test']['km']['p'])} (KM time)")
print('Tertiles (descriptive, training tertile cutoffs): ' + '; '.join(
    f"{r['group']} n = {r['n']}, deaths = {r['deaths']}, 5-year OS {fmt_os(r)}" for r in TERTILES))
for w in sorted(set(H['warnings'] + cs_['warnings'])):
    print('  lifelines note:', w)

def km_plot(labels, order, colors, short, title, note, fname, ci_show=True):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    fitters, ends = [], []
    shown = []
    for g, col, s in zip(order, colors, short):
        m = labels == g
        if not m.any():
            continue
        kmf = KaplanMeierFitter(label=f'{g}: n = {m.sum()}, deaths = {E_TE[m].sum()}').fit(T_TE[m], E_TE[m])
        kmf.plot_survival_function(ax=ax, ci_show=ci_show, ci_alpha=0.10, color=col, lw=1.0, show_censors=True,
                                   censor_styles={'ms': 4, 'marker': '|', 'mew': 0.8})
        fitters.append(kmf)
        shown.append(s)
        ends.append((kmf.survival_function_.index[-1], float(kmf.survival_function_.iloc[-1, 0]), s))
    ys = sorted(y for _, y, _ in ends)
    if all(b - a >= 0.05 for a, b in zip(ys, ys[1:])):          # direct end labels only when they do not collide
        for x_end, y_end, s in ends:
            ax.text(x_end, y_end, f' {s}', color=INK2, fontsize=8, va='center', ha='left')
    ax.set_ylim(0, 1.02)
    ax.set_xlim(left=0)
    ax.set_xlabel('Months')
    ax.set_ylabel('Overall survival')
    ax.set_title(title, loc='left', fontsize=10, color=INK)
    ax.text(0.98, 0.04, note, transform=ax.transAxes, ha='right', va='bottom', color=INK2, fontsize=8)
    ax.legend(loc='lower left', fontsize=8)
    add_at_risk_counts(*fitters, labels=shown, ax=ax, rows_to_show=['At risk'])
    savefig(fig, fname)

GROUP_TE = np.where(LP_TE > CUTOFF, 'High risk', 'Low risk')
km_plot(GROUP_TE, ['Low risk', 'High risk'], [BLUE, ORANGE], ['Low', 'High'],
        f'Test set: overall survival by risk group (training cutoff), n = {len(T_TE)}',
        f"log-rank p = {fmt_p(LR_P)}\nHR high vs low {fmt_hr(PRIMARY['hr_high_vs_low']).split(', p')[0]}",
        'km_test_median_05.png')
km_plot(TERT_TE, TERT_LABELS, TERT_COLORS, ['T1', 'T2', 'T3'],
        'Test set: overall survival by risk tertile (training cutoffs; descriptive)',
        'descriptive: no test', 'km_test_tertiles_05.png', ci_show=False)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
bins = np.linspace(min(LP_TRAIN.min(), LP_TE.min()), max(LP_TRAIN.max(), LP_TE.max()), 31)
ax.hist(LP_TRAIN, bins=bins, density=True, color=GREY, edgecolor='white', linewidth=0.8,
        label=f'Training, in-sample (n = {len(LP_TRAIN)})')
ax.hist(LP_TE, bins=bins, density=True, histtype='step', color=BLUE, lw=1.4, label=f'Test (n = {len(LP_TE)})')
ax.axvline(CUTOFF, color=INK, lw=1.0, ls='--')
ax.annotate('training cutoff', xy=(CUTOFF, 1), xycoords=('data', 'axes fraction'), xytext=(0, 3),
            textcoords='offset points', ha='center', va='bottom', color=INK2, fontsize=8)
ax.set_xlabel('Risk score (LP, training scale)')
ax.set_ylabel('Density')
ax.set_title(f'Risk score distribution: training SD {LP_TRAIN.std(ddof=1):.2f}, test SD {LP_TE.std(ddof=1):.2f}',
             loc='left', fontsize=10, color=INK, pad=16)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8)
savefig(fig, 'lp_distribution_05.png')

## 13. Negative control — the 100 permuted-label models on the true test labels (plan §9.1)
Each null model from section 10 is applied to the test set exactly as the frozen model is (z-scored with the training statistics of its own permuted fit) and scored with Harrell's C against the true test labels. An empty model has C = 0.5. Expected: permuted C centred at 0.5, and the frozen model above almost all of them. Empirical p = (1 + number of permuted C ≥ observed C) / 101; the smallest possible value is 0.0099.

TR: 10. bölümdeki 100 permütasyon modeli test setine uygulanır ve gerçek test etiketleriyle puanlanır. Etiketler karıştırıldığı için bu modeller gerçek bir şey öğrenemez; C değerleri 0,5 civarında toplanmalı. Dondurulmuş modelin C'si bu dağılımın sağında kalmalı. Permütasyon p-değeri = (1 + gözlenenden büyük ya da eşit C sayısı) / 101; en küçük değeri 0,0099.

In [ ]:
C_OBS = PRIMARY['harrell_c']
NC_TAB = NULL_TRAIN.drop(columns=['genes']).copy()
NC_TAB['empty'] = NC_TAB['n_genes'] == 0
NC_TAB['test_c'] = [harrell_c(T_TE, E_TE, lp_of(r, X_TE)) for r in NULL_MODELS]
N_GE = int((NC_TAB['test_c'] >= C_OBS).sum())
P_PERM = (1 + N_GE) / (1 + len(NC_TAB))
NEG = {'n_permutations': int(len(NC_TAB)), 'observed_c': C_OBS, 'mean_c': float(NC_TAB['test_c'].mean()),
       'sd_c': float(NC_TAB['test_c'].std(ddof=1)), 'pct_2_5': float(np.percentile(NC_TAB['test_c'], 2.5)),
       'pct_97_5': float(np.percentile(NC_TAB['test_c'], 97.5)), 'min_c': float(NC_TAB['test_c'].min()),
       'max_c': float(NC_TAB['test_c'].max()), 'share_empty_models': float(NC_TAB['empty'].mean()),
       'median_genes_nonempty': float(NC_TAB.loc[~NC_TAB['empty'], 'n_genes'].median()) if (~NC_TAB['empty']).any() else 0.0,
       'mean_c_nonempty': float(NC_TAB.loc[~NC_TAB['empty'], 'test_c'].mean()) if (~NC_TAB['empty']).any() else None,
       'n_permuted_c_ge_observed': N_GE, 'p_empirical': P_PERM}
NC_TAB.to_csv(f'{RES}/negative_control_05.csv', index=False)
write_json(NEG, f'{RES}/negative_control_05.json')
print(f"Permuted-label models on the true test labels: mean C {NEG['mean_c']:.3f} "
      f"(2.5–97.5%: {NEG['pct_2_5']:.3f}–{NEG['pct_97_5']:.3f}); empty models {NEG['share_empty_models']:.0%}; "
      f"highest {NEG['max_c']:.3f}")
print(f'Frozen model: C {C_OBS:.3f}; permuted C >= observed: {N_GE}/{len(NC_TAB)}; permutation p = {fmt_p(P_PERM)}')

fig, ax = plt.subplots(figsize=(6.4, 3.4))
lo = np.floor(min(NC_TAB['test_c'].min(), C_OBS, 0.45) * 100) / 100 - 0.01
hi = np.ceil(max(NC_TAB['test_c'].max(), C_OBS, 0.55) * 100) / 100 + 0.01
ax.hist(NC_TAB['test_c'], bins=np.arange(lo, hi + 1e-9, 0.01), color=BLUE, edgecolor='white', linewidth=1.0)
ax.axvline(0.5, color=MUTED, lw=1.0, ls='--')
ax.axvline(C_OBS, color=INK, lw=1.4)
ax.annotate('frozen model', xy=(C_OBS, 1), xycoords=('data', 'axes fraction'), xytext=(0, 3),
            textcoords='offset points', ha='center', va='bottom', color=INK, fontsize=8)
ax.set_xlabel("Harrell's C on the test set, true labels (dashed line: 0.5, no information)")
ax.set_ylabel('Permuted-label models')
ax.set_title(f'Negative control: {len(NC_TAB)} permuted-label models\nfrozen model C = {C_OBS:.3f}; '
             f'permutation p = {fmt_p(P_PERM)}', loc='left', fontsize=10, color=INK, pad=16)
savefig(fig, 'negative_control_05.png')

## 14. Sensitivity analyses (plan §8.1 and §8.4)
- **(a) z-scoring within the test set:** gene-wise mean and SD of the primary test population instead of the training values.
- **(b) correlation outliers excluded**, without re-scaling.

Two details the plan does not fix are set here and recorded in the manifest. In (a), the high/low split uses the test set's own median LP, because a numeric training cutoff has no meaning on a different scale (the same quantile rule as §10.3), and the calibration slope is not reported (§10.4). In (b), the LP values and their per-SD scaling, the training cutoff and τ are those of the primary analysis.

TR: İki duyarlılık analizi: (a) test ifadesi test setinin kendi ortalama ve SD'leriyle ölçeklenir; risk grupları testin kendi medyanıyla ayrılır (6. adımdaki dış doğrulamada yapılacağı gibi). (b) Korelasyon aykırıları çıkarılır; skorlar birincil analizden aynen alınır. Planın açıkça belirtmediği bu iki ayrıntı manifestoya yazılır.

In [ ]:
_pos = FROZEN['gene_pos']
_sd_w = X_TE[:, _pos].std(axis=0, ddof=CFG['ddof'])
WITHIN = {'gene_pos': _pos, 'coef': FROZEN['coef'], 'mean': X_TE[:, _pos].mean(axis=0), 'sd': np.where(_sd_w > 0, _sd_w, 1.0)}
LP_W = lp_of(WITHIN, X_TE)
SENS_A = evaluate(T_TE, E_TE, LP_W, cutoff=float(np.median(LP_W)), tau=TAU, boot_idx=BOOT_IDX,
                  calibration=False, ph_check=False)
SENS_A['corr_with_primary_lp'] = float(np.corrcoef(LP_W, LP_TE)[0, 1])

KEEP = ~CORR_TE
Z_PRIMARY = (LP_TE - LP_TE.mean()) / LP_TE.std(ddof=1)
BOOT_IDX_B = np.random.default_rng(S['validation_bootstrap']).integers(0, KEEP.sum(), size=(N_BOOT, KEEP.sum()))
SENS_B = evaluate(T_TE[KEEP], E_TE[KEEP], LP_TE[KEEP], z=Z_PRIMARY[KEEP], cutoff=CUTOFF, tau=TAU, boot_idx=BOOT_IDX_B,
                  calibration=True, ph_check=False)
SENS_B['excluded'] = SID_TE[~KEEP].tolist()

SENS_ROWS = []
for name, r in (('primary: training scaling, training cutoff', PRIMARY),
                ('(a) z-score within the test set, own median', SENS_A),
                ('(b) correlation outliers excluded, no re-scaling', SENS_B)):
    SENS_ROWS.append({'analysis': name, 'n': r['n'], 'deaths': r['deaths'], 'hr_per_sd': r['hr_per_sd']['hr'],
                      'hr_ci_low': r['hr_per_sd']['hr_ci_95'][0], 'hr_ci_high': r['hr_per_sd']['hr_ci_95'][1],
                      'p_wald': r['hr_per_sd']['p_wald'], 'harrell_c': r['harrell_c'],
                      'c_ci_low': r['harrell_c_ci_95'][0], 'c_ci_high': r['harrell_c_ci_95'][1], 'uno_c': r['uno_c'],
                      'cutoff': r['cutoff'], 'logrank_p': r['logrank']['p'] if r['logrank'] else None,
                      'calibration_slope': r['calibration_slope']['coef'] if 'calibration_slope' in r else None})
    print(f"{name:52s} n = {r['n']}: HR per SD {fmt_hr(r['hr_per_sd'])}; C {fmt_c(r['harrell_c'], r['harrell_c_ci_95'])}; "
          f"Uno C {r['uno_c']:.3f}; log-rank p = {fmt_p(r['logrank']['p'] if r['logrank'] else None)}")
pd.DataFrame(SENS_ROWS).to_csv(f'{RES}/sensitivity_05.csv', index=False)
write_json({'a_within_test_scaling': SENS_A, 'b_corr_outliers_excluded': SENS_B}, f'{RES}/sensitivity_05.json')
print(f"(a) correlation between the two LPs: {SENS_A['corr_with_primary_lp']:.3f}; (b) excluded: {SENS_B['excluded']}")

## 15. Red flags (plan §13) and the pre-specified conclusion (plan §12)
Red flags: any C-index above 0.90; mean C of the permuted-label models above 0.55; test C above the nested-CV mean C of step 4b by more than 0.10. If any is raised, errors are searched for before anything is reported. The conclusion wording was fixed in plan §12 before the test set was opened.

TR: Planın 13. bölümündeki kırmızı bayraklar ve 12. bölümde önceden sabitlenmiş sonuç cümlesi. Kırmızı bayrak varsa sonuç raporlanmadan önce hata aranır.

In [ ]:
C_VALUES = {'test Harrell C (primary)': PRIMARY['harrell_c'], 'test Uno C (primary)': PRIMARY['uno_c'],
            'sensitivity (a) Harrell C': SENS_A['harrell_c'], 'sensitivity (a) Uno C': SENS_A['uno_c'],
            'sensitivity (b) Harrell C': SENS_B['harrell_c'], 'sensitivity (b) Uno C': SENS_B['uno_c'],
            'highest permuted-model test C': NEG['max_c'], 'highest leakage-check fold C': LEAK_SUMMARY['max_c']}
RED_FLAGS = [f'{k} = {v:.3f} > 0.90' for k, v in C_VALUES.items() if v > 0.90]
if LEAK_MEAN_C > 0.55:
    RED_FLAGS.append(f'leakage check: mean permuted-label C = {LEAK_MEAN_C:.3f} > 0.55')
if NEG['mean_c'] > 0.55:
    RED_FLAGS.append(f"negative control: mean permuted-label test C = {NEG['mean_c']:.3f} > 0.55")
C_GAP = PRIMARY['harrell_c'] - NESTED_C
if C_GAP > 0.10:
    RED_FLAGS.append(f"test C {PRIMARY['harrell_c']:.3f} exceeds the step-4b nested-CV mean C {NESTED_C:.3f} by {C_GAP:.3f} > 0.10")
EXTRA_WARNINGS = []
if H['hr_ci_95'][1] < 1:
    EXTRA_WARNINGS.append('HR per SD is significantly below 1: association in the opposite direction; check the sign '
                          'convention before any interpretation.')

SUPPORT = {'Harrell C 95% CI above 0.5': bool(PRIMARY['harrell_c_ci_95'][0] > 0.5),
           'log-rank p < 0.05': bool(LR_P is not None and LR_P < 0.05),
           'permutation p < 0.05': bool(P_PERM < 0.05)}
CONCLUSION = ('Internally validated (H1 met)' if H1_MET else
              'Not internally validated: H1 not met (a negative or inconclusive result, reported with the same detail)')

print('Red flags (plan §13):', 'none' if not RED_FLAGS else len(RED_FLAGS))
for rf in RED_FLAGS:
    print('  RED FLAG:', rf)
for w in EXTRA_WARNINGS:
    print('  WARNING:', w)
print(f"H1 (primary hypothesis): HR per SD {fmt_hr(H)} -> {'MET' if H1_MET else 'NOT MET'}")
for k, v in SUPPORT.items():
    print(f"  supporting: {k}: {'yes' if v else 'no'}")
print(f"Test C minus nested-CV mean C (step 4b, {NESTED_C:.3f}): {C_GAP:+.3f}")
print('Conclusion (wording of plan §12):', CONCLUSION)
if RED_FLAGS:
    print('\nA red flag was raised: do not report these results before the pipeline has been searched for errors.')

## 16. Save the results, the manifest and the summary
Patient-level test scores (`test_lp_05.csv`, used again in step 7), all metrics, figures and `manifest_05.json` with the SHA-256 of every input and output. The key results of the first complete run are written once to `first_results_05.json` and never overwritten; a later run is compared with it (plan §17).

TR: Bütün çıktıları `results/05/` ve `figures/05/` klasörlerine yazar; `manifest_05.json` girdilerin ve çıktıların SHA-256 özetlerini kaydeder. İlk tam çalıştırmanın ana sonuçları `first_results_05.json` dosyasına bir kez yazılır ve bir daha değiştirilmez. En alttaki özeti bana yapıştır.

In [ ]:
pd.DataFrame({'sample_id': SID_TE, 'lp': LP_TE, 'lp_per_sd': Z_PRIMARY,
              'risk_group': np.where(LP_TE > CUTOFF, 'high', 'low'), 'tertile': TERT_TE,
              'lp_within_test_scaling': LP_W, FLAG_S: CORR_TE}).to_csv(f'{RES}/test_lp_05.csv', index=False)
def _km_flat(grouping, r):
    return {'grouping': grouping, 'group': r['group'], 'n': r['n'], 'deaths': r['deaths'],
            'median_os_months': r['median_os_months'], 'median_ci_low': r['median_ci_95'][0],
            'median_ci_high': r['median_ci_95'][1], 'os_5y': r['os_5y'], 'os_5y_ci_low': r['os_5y_ci_95'][0],
            'os_5y_ci_high': r['os_5y_ci_95'][1]}

pd.DataFrame([_km_flat('training median cutoff', r) for r in PRIMARY['groups']]
             + [_km_flat('training tertile cutoffs (descriptive)', r) for r in TERTILES]).to_csv(
    f'{RES}/km_test_05.csv', index=False)
VALIDATION = {'population': {'n': PRIMARY['n'], 'deaths': PRIMARY['deaths'], 'excluded_by_primary_qc': EXCLUDED_TE,
                             'corr_outliers_kept': int(CORR_TE.sum())},
              'scaling': 'training means and SDs from model_04.json', 'primary': PRIMARY, 'tertiles_descriptive': TERTILES,
              'H1': {'definition': IV['H1'], 'met': H1_MET}, 'supporting_evidence': SUPPORT,
              'permutation_p': P_PERM, 'red_flags': RED_FLAGS, 'extra_warnings': EXTRA_WARNINGS,
              'test_c_minus_nested_cv_mean_c': C_GAP, 'conclusion': CONCLUSION}
write_json(VALIDATION, f'{RES}/internal_validation_05.json')

KEY = {'code_version': CODE_VERSION, 'model_sha256': MODEL_SHA, 'n': PRIMARY['n'], 'deaths': PRIMARY['deaths'],
       'hr_per_sd': H['hr'], 'hr_per_sd_ci_95': H['hr_ci_95'], 'p_wald': H['p_wald'], 'harrell_c': PRIMARY['harrell_c'],
       'harrell_c_ci_95': PRIMARY['harrell_c_ci_95'], 'uno_c': PRIMARY['uno_c'], 'logrank_p': LR_P,
       'permutation_p': P_PERM, 'h1_met': H1_MET}
FIRST_PATH = f'{RES}/first_results_05.json'

def _same(a, b):
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, float) or isinstance(b, float):
        return a is not None and b is not None and bool(np.isclose(a, b, rtol=1e-9, atol=1e-12))
    return a == b

if not os.path.exists(FIRST_PATH):
    write_json({'note': 'Written once, at the first complete run of step 5; never overwritten (plan §17).',
                'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'), **KEY}, FIRST_PATH)
    FIRST_STATUS = 'first complete run: first_results_05.json written'
else:
    with open(FIRST_PATH) as f:
        _first = json.load(f)
    _diff = [k for k in KEY if k != 'code_version' and not _same(_first.get(k), to_builtin(KEY[k]))]
    FIRST_STATUS = ('identical to the first run' if not _diff else
                    f"DIFFERS from the first run in {_diff}; first_results_05.json kept unchanged (plan §17: report both)")

L = []
L.append(f"Step 5 summary (plan v{PLAN['version']}, code {CODE_VERSION}, {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M} UTC)")
L.append(f"Reproducibility: step-4b fit reproduced ({len(REPRO['entrez'])} genes, lambda {REPRO['lambda']:.5g}, max |coef diff| {MAX_DCOEF:.1e})")
L.append(f"Leakage check: mean C {LEAK_MEAN_C:.3f} ({N_LEAK} permutations x {LEAK_FOLDS} folds, folds {LEAK_SUMMARY['min_c']:.3f}–{LEAK_SUMMARY['max_c']:.3f}, "
         f"empty models {LEAK_SUMMARY['share_empty_models']:.0%}) -> {'PASSED' if LEAKAGE_PASSED else 'FAILED'}")
L.append(f"Frozen model: sha256 {MODEL_SHA[:12]} = manifest_04b = progress_step4b; {len(FROZEN['coef'])} genes; cutoff {CUTOFF:.4f}")
L.append(f"Null models: {N_PERM} fixed before the test set was opened; empty {NEG['share_empty_models']:.0%}")
L.append(f"Test set: {len(te)} patients ({int(te[EVENT].sum())} deaths); QC excluded {len(EXCLUDED_TE)} -> {PRIMARY['n']} patients, {PRIMARY['deaths']} deaths; median follow-up {FU_TE:.1f} mo")
L.append(f"H1  HR per SD {fmt_hr(H)} -> {'MET' if H1_MET else 'NOT MET'}")
L.append(f"Harrell C {fmt_c(PRIMARY['harrell_c'], PRIMARY['harrell_c_ci_95'])}; Uno C {PRIMARY['uno_c']:.3f} (tau {TAU:g} months)")
for row in PRIMARY['groups']:
    L.append(f"KM {row['group']:4s}: n {row['n']}, deaths {row['deaths']}, median OS {fmt_median(row)}, 5-year OS {fmt_os(row)}")
L.append(f"Log-rank p = {fmt_p(LR_P)}; HR high vs low {fmt_hr(PRIMARY['hr_high_vs_low'])}")
L.append(f"Calibration slope {cs_['coef']:.2f} ({cs_['coef_ci_95'][0]:.2f}–{cs_['coef_ci_95'][1]:.2f}); "
         f"PH test p = {fmt_p(PRIMARY['ph_test']['rank']['p'])} (rank), {fmt_p(PRIMARY['ph_test']['km']['p'])} (KM)")
L.append('Tertiles: ' + '; '.join(f"{r['group']} n {r['n']}, deaths {r['deaths']}, 5y OS {fmt_os(r)}" for r in TERTILES))
L.append(f"Negative control: permuted test C mean {NEG['mean_c']:.3f} ({NEG['pct_2_5']:.3f}–{NEG['pct_97_5']:.3f}), max {NEG['max_c']:.3f}; "
         f"permutation p = {fmt_p(P_PERM)}")
L.append(f"Sensitivity (a) within-test z-score: HR per SD {fmt_hr(SENS_A['hr_per_sd'])}; C {SENS_A['harrell_c']:.3f}")
L.append(f"Sensitivity (b) corr outliers excluded (n {SENS_B['n']}): HR per SD {fmt_hr(SENS_B['hr_per_sd'])}; C {SENS_B['harrell_c']:.3f}")
L.append(f"Supporting: C CI > 0.5 {'yes' if SUPPORT['Harrell C 95% CI above 0.5'] else 'no'}; log-rank p < 0.05 "
         f"{'yes' if SUPPORT['log-rank p < 0.05'] else 'no'}; permutation p < 0.05 {'yes' if SUPPORT['permutation p < 0.05'] else 'no'}")
L.append(f"Test C - nested-CV C: {C_GAP:+.3f}; red flags: {'none' if not RED_FLAGS else ' | '.join(RED_FLAGS)}")
if EXTRA_WARNINGS:
    L.append('Warnings: ' + ' | '.join(EXTRA_WARNINGS))
L.append(f'Conclusion (plan §12): {CONCLUSION}')
L.append(f'First-run record: {FIRST_STATUS}; test-set openings logged: {N_PREVIOUS_OPENINGS + 1}')
SUMMARY = '\n'.join(L)
with open(f'{RES}/summary_05.txt', 'w') as f:
    f.write(SUMMARY + '\n')

OUTPUT_FILES = ['leakage_check_05.csv', 'leakage_check_05.json', 'frozen_model_check_05.json', 'null_models_05.csv',
                'test_lp_05.csv', 'km_test_05.csv', 'internal_validation_05.json', 'negative_control_05.csv',
                'negative_control_05.json', 'sensitivity_05.csv', 'sensitivity_05.json', 'first_results_05.json',
                'summary_05.txt']
FIGURES = ['leakage_check_05.png', 'km_test_median_05.png', 'km_test_tertiles_05.png', 'lp_distribution_05.png',
           'negative_control_05.png']
RUNTIME['total_s'] = round(time.time() - T_START, 1)
MANIFEST = {
    'step': '05', 'notebook': '05_internal_validation.ipynb', 'code_version': CODE_VERSION,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'plan': {'version': PLAN['version'], 'plan_json_sha256': LOCK['plan_json_sha256'], 'plan_md_sha256': LOCK['plan_md_sha256'],
             'lock_file_sha256': LOCK_SHA, 'amendments': len(AMENDMENTS), 'sections_implemented': ['8', '9', '13']},
    'frozen_model': {'file': 'results/04/model_04.json', 'sha256': MODEL_SHA, 'matches': list(HASH_REFS),
                     'n_genes': int(len(FROZEN['coef'])), 'lambda': FROZEN['lambda'], 'cutoff': CUTOFF,
                     'reproduced_from_training_data': True, 'max_abs_coef_difference': MAX_DCOEF},
    'inputs_verified_against_lock': INPUT_SHA,
    'data_access': {
        'before_test_opening': ['training rows of data/clean/clinical_clean_all.csv', 'data/clean/split_GSE181063.csv',
                                'training rows of data/processed/qc_samples.csv', 'data/processed/expr_GSE181063_train.parquet',
                                'data/processed/gene_table_common.csv', 'results/04 plan, lock, amendments, manifest_04b.json',
                                'results/04/model_04.json (after the leakage check)'],
        'gates_before_test_opening': list(GATES), 'null_models_fixed_utc': NULL_DONE_UTC,
        'test_opened_utc': TEST_OPENED_UTC,
        'opened_after_gates': ['test rows of data/clean/clinical_clean_all.csv', 'test rows of data/processed/qc_samples.csv',
                               REL_TEST],
        'test_access_log': 'results/05/test_access_log_05.jsonl', 'test_openings_logged': N_PREVIOUS_OPENINGS + 1,
        'hashed_only_never_opened': {k: v['file'] for k, v in INPUT_SHA.items() if k not in ACCESS}},
    'populations': {
        'training': {'patients': int(len(T)), 'deaths': int(E.sum()), 'excluded_by_primary_qc': EXCLUDED},
        'test': {'patients_before_qc': int(len(te)), 'deaths_before_qc': int(te[EVENT].sum()),
                 'excluded_by_primary_qc': EXCLUDED_TE, 'patients': PRIMARY['n'], 'deaths': PRIMARY['deaths'],
                 'corr_outliers_kept': int(CORR_TE.sum()), 'median_follow_up_months_reverse_km': FU_TE}},
    'leakage_check': LEAK_SUMMARY,
    'internal_validation': {
        'H1_met': H1_MET, 'hr_per_sd': H['hr'], 'hr_per_sd_ci_95': H['hr_ci_95'], 'p_wald': H['p_wald'],
        'harrell_c': PRIMARY['harrell_c'], 'harrell_c_ci_95': PRIMARY['harrell_c_ci_95'], 'uno_c': PRIMARY['uno_c'],
        'uno_tau_months': TAU, 'logrank_p': LR_P, 'hr_high_vs_low': HRG['hr'],
        'hr_high_vs_low_ci_95': HRG['hr_ci_95'], 'calibration_slope': cs_['coef'],
        'calibration_slope_ci_95': cs_['coef_ci_95'], 'ph_test_p': {k: v['p'] for k, v in PRIMARY['ph_test'].items()},
        'groups': PRIMARY['groups']},
    'negative_control': NEG,
    'sensitivity': SENS_ROWS,
    'supporting_evidence': SUPPORT, 'conclusion': CONCLUSION, 'red_flags': RED_FLAGS, 'extra_warnings': EXTRA_WARNINGS,
    'test_c_minus_nested_cv_mean_c': C_GAP, 'first_results_status': FIRST_STATUS,
    'implementation_notes': [
        'Order: integrity checks; reproducibility check (step-4b fit re-run on the true training labels, compared with manifest_04b.json); leakage check (gate 1); frozen-model hash and content check (gate 2); 100 null models fitted and saved; test set opened (gate 3).',
        'Permutations: numpy default_rng(seed).permutation over the primary training patients sorted by sample_id; (time, event) pairs moved together.',
        'Leakage check: permutation i with label seed SEED + 3000 + i; outer StratifiedKFold(5, shuffle=True, random_state=SEED + 3100 + i) on the permuted event; full step-4b procedure in each outer training fold with inner CV seed SEED + 3200 + 10*i + j; Harrell C of the lambda_min model on the outer test fold (permuted labels); an empty model gives 0.5.',
        'Null models: label seed SEED + 1000 + i, CV seed SEED + 2000 + i, all 373 training patients; lambda_min model; each applied to the test set with the training means and SDs of its own permuted fit; Harrell C against the true test labels.',
        'Frozen model check: SHA-256 against manifest_04b.json (frozen_model and outputs) and progress_step4b.md; training means, SDs, median cutoff, tertile cutoffs and LP mean/SD recomputed (rtol 1e-9, atol 1e-12); re-fit vs file: identical genes, coefficients rtol 1e-6 / atol 1e-9, penalty rtol 1e-9, same grid position.',
        'Validation Cox models: lifelines CoxPHFitter (Efron ties, no penalty). HR per SD: LP standardized with the mean and SD (ddof 1) of the analysis population. Calibration slope: coefficient of the raw LP (training scale). PH check: lifelines proportional_hazard_test on the HR-per-SD model, rank and KM time transforms.',
        "Harrell C: scikit-survival concordance_index_censored. Bootstrap: 1,000 resamples of patients with replacement, numpy default_rng(20261415).integers, all index vectors drawn at once; percentile interval.",
        "Uno C: scikit-survival concordance_index_ipcw with the analysis population as both the censoring and the evaluation sample; tau = 60 months unless fewer than 20% have follow-up >= 60 months.",
        'Kaplan-Meier: lifelines defaults (exponential Greenwood CI); 5-year OS = KM at 60 months; high risk = LP > training cutoff (median in-sample training LP); tertiles from the training tertile cutoffs (T1: LP <= q1; T2: q1 < LP <= q2; T3: LP > q2), descriptive.',
        'Sensitivity (a), not fixed by the plan: high/low split at the test population\'s own median LP (as in plan section 10.3); calibration slope not reported (section 10.4); same bootstrap index vectors as the primary analysis.',
        'Sensitivity (b), not fixed by the plan: LP values and their per-SD scaling, training cutoff and tau of the primary analysis; bootstrap default_rng(20261415) on the subset size.',
        'Empirical permutation p = (1 + number of permuted test C >= observed test C) / (1 + 100).'],
    'environment': VERSIONS, 'environment_step4b': M4B['environment'], 'parallel_workers': N_JOBS,
    'runtime_seconds_this_run': RUNTIME,
    'outputs': {**{f'results/05/{f}': sha256_file(f'{RES}/{f}') for f in OUTPUT_FILES},
                **{f'figures/05/{f}': sha256_file(f'{FIG}/{f}') for f in FIGURES}},
}
write_json(MANIFEST, f'{RES}/manifest_05.json')
print(f"Saved {len(OUTPUT_FILES)} result files, {len(FIGURES)} figures and results/05/manifest_05.json "
      f"(total run time this session {fmt_min(RUNTIME['total_s'])}).\n")
print('=' * 100)
print(SUMMARY)
print('=' * 100)

TR: Bitti. Yukarıdaki iki çizgi arasındaki özeti sohbete yapıştır. Sonra **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** ile oturumu kapat.